In [1]:
import re
from eppy.modeleditor import IDF

In [2]:
IDD_FILE   = r"C:\EnergyPlusV25-1-0\Energy+.idd"
INPUT_IDF  = r"C:\Users\yhw15\Documents\1947-RP\ASHRAE901_OfficeMedium_STD2022\ASHRAE901_OfficeMedium_STD2022_NewYork.idf"

In [3]:
IDF.setiddname(IDD_FILE)
idf = IDF(INPUT_IDF)
print("IDF loaded successfully")
AHU_LIST = ["bot", "mid", "top"]

IDF loaded successfully


# Phase 1: Delete DX Cooling and Heating

## Current Status Check

In [4]:
# 1) CoilSystem:Cooling:DX
objs = idf.idfobjects["COILSYSTEM:COOLING:DX"]
print(f"\n[CoilSystem:Cooling:DX] — Total: {len(objs)}")
for o in objs:
    print(f"  • {o.Name}")

# 2) Coil:Cooling:DX:TwoSpeed
objs = idf.idfobjects["COIL:COOLING:DX:TWOSPEED"]
print(f"\n[Coil:Cooling:DX:TwoSpeed] — Total: {len(objs)}")
for o in objs:
    print(f"  • {o.Name}")
    curve_fields = [
        "Total_Cooling_Capacity_Function_of_Temperature_Curve_Name",
        "Total_Cooling_Capacity_Function_of_Flow_Fraction_Curve_Name",
        "Energy_Input_Ratio_Function_of_Temperature_Curve_Name",
        "Energy_Input_Ratio_Function_of_Flow_Fraction_Curve_Name",
        "Part_Load_Fraction_Correlation_Curve_Name",
        "Low_Speed_Total_Cooling_Capacity_Function_of_Temperature_Curve_Name",
        "Low_Speed_Energy_Input_Ratio_Function_of_Temperature_Curve_Name",
    ]
    for cf in curve_fields:
        try:
            val = getattr(o, cf, "")
            if val:
                print(f"      ↳ {cf}: {val}")
        except:
            pass

# 3) Coil:Heating:Fuel
objs = idf.idfobjects["COIL:HEATING:FUEL"]
print(f"\n[Coil:Heating:Fuel] — Total: {len(objs)}")
for o in objs:
    print(f"  • {o.Name}")

# 4) DX performance curves
for curve_type in ["CURVE:BIQUADRATIC", "CURVE:QUADRATIC"]:
    objs = idf.idfobjects[curve_type]
    print(f"\n[{curve_type}] — Total: {len(objs)}")
    for o in objs:
        print(f"  • {o.Name}")

# 5) DX OutdoorAir:Node
objs = idf.idfobjects["OUTDOORAIR:NODE"]
print(f"\n[OutdoorAir:Node] — Total: {len(objs)}")
for o in objs:
    print(f"  • {o.Name}  →  {'OUTDOORAIR:NODE' if 'CoolC' in str(o.Name) else 'Unknown'}")

# 6) AHU Main Branch
print(f"\n{'=' * 65}")
print("  Current AirLoop Main Branch Structure")
print("=" * 65)
branches = idf.idfobjects["BRANCH"]
for br in branches:
    if "Air Loop Main Branch" in str(br.Name):
        print(f"\n  Branch: {br.Name}")
        idx = 2
        comp_num = 1
        while idx + 3 < len(br.fieldnames):
            comp_type = br[br.fieldnames[idx]]
            comp_name = br[br.fieldnames[idx + 1]]
            in_node   = br[br.fieldnames[idx + 2]]
            out_node  = br[br.fieldnames[idx + 3]]
            if comp_type and str(comp_type).strip():
                print(f"    [{comp_num}] {comp_type}: {comp_name}")
                print(f"        IN:  {in_node}")
                print(f"        OUT: {out_node}")
                comp_num += 1
            idx += 4


[CoilSystem:Cooling:DX] — Total: 3
  • PACU_VAV_bot Cooling Coil
  • PACU_VAV_mid Cooling Coil
  • PACU_VAV_top Cooling Coil

[Coil:Cooling:DX:TwoSpeed] — Total: 3
  • PACU_VAV_bot Cooling Coil
      ↳ Total_Cooling_Capacity_Function_of_Temperature_Curve_Name: LgDXalt_CapFT
      ↳ Total_Cooling_Capacity_Function_of_Flow_Fraction_Curve_Name: LgDXalt_CapFF
      ↳ Energy_Input_Ratio_Function_of_Temperature_Curve_Name: LgDXalt_EIRFT
      ↳ Energy_Input_Ratio_Function_of_Flow_Fraction_Curve_Name: LgDXalt_EIRFF
      ↳ Part_Load_Fraction_Correlation_Curve_Name: LgDXalt_PLR
      ↳ Low_Speed_Total_Cooling_Capacity_Function_of_Temperature_Curve_Name: LgDXalt_CapFT
      ↳ Low_Speed_Energy_Input_Ratio_Function_of_Temperature_Curve_Name: LgDXalt_EIRFT
  • PACU_VAV_mid Cooling Coil
      ↳ Total_Cooling_Capacity_Function_of_Temperature_Curve_Name: LgDXalt_CapFT
      ↳ Total_Cooling_Capacity_Function_of_Flow_Fraction_Curve_Name: LgDXalt_CapFF
      ↳ Energy_Input_Ratio_Function_of_Temperature

## Gathering all DX

In [5]:
dx_curve_names = set()

for coil in idf.idfobjects["COIL:COOLING:DX:TWOSPEED"]:
    curve_fields = [
        "Total_Cooling_Capacity_Function_of_Temperature_Curve_Name",
        "Total_Cooling_Capacity_Function_of_Flow_Fraction_Curve_Name",
        "Energy_Input_Ratio_Function_of_Temperature_Curve_Name",
        "Energy_Input_Ratio_Function_of_Flow_Fraction_Curve_Name",
        "Part_Load_Fraction_Correlation_Curve_Name",
        "Low_Speed_Total_Cooling_Capacity_Function_of_Temperature_Curve_Name",
        "Low_Speed_Energy_Input_Ratio_Function_of_Temperature_Curve_Name",
    ]
    for cf in curve_fields:
        try:
            val = getattr(coil, cf, "")
            if val and str(val).strip():
                dx_curve_names.add(str(val).strip())
        except:
            pass

print(f"DX performance curves in total: {len(dx_curve_names)}:")
for name in sorted(dx_curve_names):
    print(f"  • {name}")


DX performance curves in total: 5:
  • LgDXalt_CapFF
  • LgDXalt_CapFT
  • LgDXalt_EIRFF
  • LgDXalt_EIRFT
  • LgDXalt_PLR


## Delete CoilSystem:Cooling:DX (3)

In [6]:
removed = []
for obj in idf.idfobjects["COILSYSTEM:COOLING:DX"][:]:
    removed.append(obj.Name)
    idf.removeidfobject(obj)

print(f"CoilSystem:Cooling:DX Deleted — {len(removed)}:")
for name in removed:
    print(f"   ✗ {name}")
print(f"   Validation: Remaining {len(idf.idfobjects['COILSYSTEM:COOLING:DX'])} objects")

CoilSystem:Cooling:DX Deleted — 3:
   ✗ PACU_VAV_bot Cooling Coil
   ✗ PACU_VAV_mid Cooling Coil
   ✗ PACU_VAV_top Cooling Coil
   Validation: Remaining 0 objects


## Delete Coil:Cooling:DX:TwoSpeed (3)

In [7]:
removed = []
for obj in idf.idfobjects["COIL:COOLING:DX:TWOSPEED"][:]:
    removed.append(obj.Name)
    idf.removeidfobject(obj)

print(f"Coil:Cooling:DX:TwoSpeed Deleted — {len(removed)}:")
for name in removed:
    print(f"   ✗ {name}")
print(f"   Validation: Remaining {len(idf.idfobjects['COIL:COOLING:DX:TWOSPEED'])} objects")

Coil:Cooling:DX:TwoSpeed Deleted — 3:
   ✗ PACU_VAV_bot Cooling Coil
   ✗ PACU_VAV_mid Cooling Coil
   ✗ PACU_VAV_top Cooling Coil
   Validation: Remaining 0 objects


## Delete Coil:Heating:Fuel (3)

In [8]:
removed = []
for obj in idf.idfobjects["COIL:HEATING:FUEL"][:]:
    removed.append(obj.Name)
    idf.removeidfobject(obj)

print(f"Coil:Heating:Fuel Deleted — {len(removed)}:")
for name in removed:
    print(f"   ✗ {name}")
print(f"   Validation: Remaining {len(idf.idfobjects['COIL:HEATING:FUEL'])} objects")

elec_coils = idf.idfobjects["COIL:HEATING:ELECTRIC"]
print(f"\n Coil:Heating:Electric Remaining: {len(elec_coils)}")

Coil:Heating:Fuel Deleted — 3:
   ✗ PACU_VAV_bot Heating Coil
   ✗ PACU_VAV_mid Heating Coil
   ✗ PACU_VAV_top Heating Coil
   Validation: Remaining 0 objects

 Coil:Heating:Electric Remaining: 15


## Delete DX Curve

In [9]:
removed_curves = []

for curve_type in ["CURVE:BIQUADRATIC", "CURVE:QUADRATIC"]:
    for obj in idf.idfobjects[curve_type][:]:
        if str(obj.Name).strip() in dx_curve_names:
            removed_curves.append((curve_type, obj.Name))
            idf.removeidfobject(obj)

print(f"DX Performance Curves Deleted — {len(removed_curves)}:")
for ctype, cname in removed_curves:
    print(f"   ✗ [{ctype}] {cname}")

for curve_type in ["CURVE:BIQUADRATIC", "CURVE:QUADRATIC"]:
    remaining = len(idf.idfobjects[curve_type])
    print(f"   Validation: {curve_type} Remaining {remaining} objects")

DX Performance Curves Deleted — 5:
   ✗ [CURVE:BIQUADRATIC] LgDXalt_CapFT
   ✗ [CURVE:BIQUADRATIC] LgDXalt_EIRFT
   ✗ [CURVE:QUADRATIC] LgDXalt_CapFF
   ✗ [CURVE:QUADRATIC] LgDXalt_EIRFF
   ✗ [CURVE:QUADRATIC] LgDXalt_PLR
   Validation: CURVE:BIQUADRATIC Remaining 0 objects
   Validation: CURVE:QUADRATIC Remaining 0 objects


## Delete OutDoorAir

In [10]:
removed_nodes = []

for obj in idf.idfobjects["OUTDOORAIR:NODE"][:]:
    name = str(obj.Name)
    if "CoolC" in name:
        removed_nodes.append(name)
        idf.removeidfobject(obj)

print(f"DX OutdoorAir:Node Deleted — {len(removed_nodes)}:")
for name in removed_nodes:
    print(f"   ✗ {name}")

remaining = idf.idfobjects["OUTDOORAIR:NODE"]
print(f"   Validation: OutdoorAir:Node Remaining {len(remaining)} objects (should be OA system nodes)")

DX OutdoorAir:Node Deleted — 6:
   ✗ PACU_VAV_bot_CoolC Cond OA node
   ✗ PACU_VAV_bot_CoolCOA Ref node
   ✗ PACU_VAV_mid_CoolC Cond OA node
   ✗ PACU_VAV_mid_CoolCOA Ref node
   ✗ PACU_VAV_top_CoolC Cond OA node
   ✗ PACU_VAV_top_CoolCOA Ref node
   Validation: OutdoorAir:Node Remaining 0 objects (should be OA system nodes)


## Update Main Branch

In [11]:
print("=" * 65)
print("  Update AirLoop Main Branch")
print("=" * 65)

REMOVED_TYPES = {
    "CoilSystem:Cooling:DX",
    "Coil:Heating:Fuel",
}

for br in idf.idfobjects["BRANCH"]:
    if "Air Loop Main Branch" not in str(br.Name):
        continue

    print(f"\n  Branch: {br.Name}")
    fieldnames = br.fieldnames
    idx = 2

    while idx + 3 < len(fieldnames):
        comp_type_field = fieldnames[idx]
        comp_name_field = fieldnames[idx + 1]
        inlet_field     = fieldnames[idx + 2]
        outlet_field    = fieldnames[idx + 3]

        comp_type = str(br[comp_type_field]).strip()

        if comp_type in REMOVED_TYPES:
            old_name = br[comp_name_field]
            in_node  = br[inlet_field]
            out_node = br[outlet_field]

            br[comp_type_field] = ""
            br[comp_name_field] = ""

            print(f"    Deleted: {comp_type} → {old_name}")
            print(f"    Node retained: {in_node} → {out_node}")

        idx += 4

  Update AirLoop Main Branch

  Branch: PACU_VAV_bot Air Loop Main Branch

  Branch: PACU_VAV_mid Air Loop Main Branch

  Branch: PACU_VAV_top Air Loop Main Branch


# Phase 2: Add Water Coil and Controller

In [13]:
br_sample = None
for br in idf.idfobjects["BRANCH"]:
    if "Air Loop Main Branch" in str(br.Name):
        br_sample = br
        break

print("Branch fieldnames (eppy):")
for i, fn in enumerate(br_sample.fieldnames):
    val = br_sample[fn]
    print(f"  [{i:2d}] {fn:45s} = {val}")

Branch fieldnames (eppy):
  [ 0] key                                           = Branch
  [ 1] Name                                          = PACU_VAV_bot Air Loop Main Branch
  [ 2] Pressure_Drop_Curve_Name                      = 
  [ 3] Component_1_Object_Type                       = AirLoopHVAC:OutdoorAirSystem
  [ 4] Component_1_Name                              = PACU_VAV_bot_OA
  [ 5] Component_1_Inlet_Node_Name                   = PACU_VAV_bot Supply Equipment Inlet Node
  [ 6] Component_1_Outlet_Node_Name                  = PACU_VAV_bot_OA-PACU_VAV_bot_CoolCNode
  [ 7] Component_2_Object_Type                       = CoilSystem:Cooling:DX
  [ 8] Component_2_Name                              = PACU_VAV_bot Cooling Coil
  [ 9] Component_2_Inlet_Node_Name                   = PACU_VAV_bot_OA-PACU_VAV_bot_CoolCNode
  [10] Component_2_Outlet_Node_Name                  = PACU_VAV_bot_CoolC-PACU_VAV_bot_HeatCNode
  [11] Component_3_Object_Type                       = Coil:Heating:Fuel


## Define Nodes

In [ ]:
AHU_LIST = ["bot", "mid", "top"]

ahu_nodes = {}

for br in idf.idfobjects["BRANCH"]:
    if "Air Loop Main Branch" not in str(br.Name):
        continue

    for x in AHU_LIST:
        if f"PACU_VAV_{x}" in str(br.Name):
            ahu_nodes[x] = {
                # Component 2
                "cool_air_inlet":  str(br.Component_2_Inlet_Node_Name).strip(),
                "cool_air_outlet": str(br.Component_2_Outlet_Node_Name).strip(),
                # Component 3
                "heat_air_inlet":  str(br.Component_3_Inlet_Node_Name).strip(),
                "heat_air_outlet": str(br.Component_3_Outlet_Node_Name).strip(),
                # Water nodes to be created
                "cool_water_inlet":  f"PACU_VAV_{x} CoolC Water Inlet Node",
                "cool_water_outlet": f"PACU_VAV_{x} CoolC Water Outlet Node",
                "heat_water_inlet":  f"PACU_VAV_{x} HeatC Water Inlet Node",
                "heat_water_outlet": f"PACU_VAV_{x} HeatC Water Outlet Node",
                # Names for new coils and controllers
                "cw_coil_name":      f"PACU_VAV_{x} CW Cooling Coil",
                "hw_coil_name":      f"PACU_VAV_{x} HW Heating Coil",
                "cw_ctrl_name":      f"PACU_VAV_{x} CW CoolC Controller",
                "hw_ctrl_name":      f"PACU_VAV_{x} HW HeatC Controller",
                "ctrl_list_name":    f"PACU_VAV_{x}_Water_Coil_Controllers",
            }
            break

print("=" * 70)
print("  AHU Nodes and New Component Names")
print("=" * 70)
for x in AHU_LIST:
    n = ahu_nodes[x]
    print(f"\n  PACU_VAV_{x}:")
    print(f"    CW Cooling Coil: {n['cw_coil_name']}")
    print(f"      Air:   {n['cool_air_inlet']}  →  {n['cool_air_outlet']}")
    print(f"      Water: {n['cool_water_inlet']}  →  {n['cool_water_outlet']}")
    print(f"    HW Heating Coil: {n['hw_coil_name']}")
    print(f"      Air:   {n['heat_air_inlet']}  →  {n['heat_air_outlet']}")
    print(f"      Water: {n['heat_water_inlet']}  →  {n['heat_water_outlet']}")
    print(f"    CW Controller: {n['cw_ctrl_name']}")
    print(f"      Sensor (air out): {n['cool_air_outlet']}")
    print(f"      Actuator (water in): {n['cool_water_inlet']}")
    print(f"    HW Controller: {n['hw_ctrl_name']}")
    print(f"      Sensor (air out): {n['heat_air_outlet']}")
    print(f"      Actuator (water in): {n['heat_water_inlet']}")

  AHU Nodes and New Component Names

  PACU_VAV_bot:
    CW Cooling Coil: PACU_VAV_bot CW Cooling Coil
      Air:   PACU_VAV_bot_OA-PACU_VAV_bot_CoolCNode  →  PACU_VAV_bot_CoolC-PACU_VAV_bot_HeatCNode
      Water: PACU_VAV_bot CoolC Water Inlet Node  →  PACU_VAV_bot CoolC Water Outlet Node
    HW Heating Coil: PACU_VAV_bot HW Heating Coil
      Air:   PACU_VAV_bot_CoolC-PACU_VAV_bot_HeatCNode  →  PACU_VAV_bot_HeatC-PACU_VAV_bot FanNode
      Water: PACU_VAV_bot HeatC Water Inlet Node  →  PACU_VAV_bot HeatC Water Outlet Node
    CW Controller: PACU_VAV_bot CW CoolC Controller
      Sensor (air out): PACU_VAV_bot_CoolC-PACU_VAV_bot_HeatCNode
      Actuator (water in): PACU_VAV_bot CoolC Water Inlet Node
    HW Controller: PACU_VAV_bot HW HeatC Controller
      Sensor (air out): PACU_VAV_bot_HeatC-PACU_VAV_bot FanNode
      Actuator (water in): PACU_VAV_bot HeatC Water Inlet Node

  PACU_VAV_mid:
    CW Cooling Coil: PACU_VAV_mid CW Cooling Coil
      Air:   PACU_VAV_mid_OA-PACU_VAV_mid_C

## Add Coil:Cooling:Water (3)

In [16]:
for x in AHU_LIST:
    n = ahu_nodes[x]

    coil = idf.newidfobject("COIL:COOLING:WATER")
    coil.Name                              = n["cw_coil_name"]
    coil.Availability_Schedule_Name        = "ALWAYS_ON"
    coil.Design_Water_Flow_Rate            = "autosize"
    coil.Design_Air_Flow_Rate              = "autosize"
    coil.Design_Inlet_Water_Temperature    = "autosize"
    coil.Design_Inlet_Air_Temperature      = "autosize"
    coil.Design_Outlet_Air_Temperature     = "autosize"
    coil.Design_Inlet_Air_Humidity_Ratio   = "autosize"
    coil.Design_Outlet_Air_Humidity_Ratio  = "autosize"
    coil.Water_Inlet_Node_Name             = n["cool_water_inlet"]
    coil.Water_Outlet_Node_Name            = n["cool_water_outlet"]
    coil.Air_Inlet_Node_Name               = n["cool_air_inlet"]
    coil.Air_Outlet_Node_Name              = n["cool_air_outlet"]
    coil.Type_of_Analysis                  = "SimpleAnalysis"
    coil.Heat_Exchanger_Configuration      = "CrossFlow"

    print(f"  ✅ {coil.Name}")
    print(f"     Air:   {n['cool_air_inlet']}  →  {n['cool_air_outlet']}")
    print(f"     Water: {n['cool_water_inlet']}  →  {n['cool_water_outlet']}")

print(f"\n  ✅ Coil:Cooling:Water: {len(idf.idfobjects['COIL:COOLING:WATER'])}")

  ✅ PACU_VAV_bot CW Cooling Coil
     Air:   PACU_VAV_bot_OA-PACU_VAV_bot_CoolCNode  →  PACU_VAV_bot_CoolC-PACU_VAV_bot_HeatCNode
     Water: PACU_VAV_bot CoolC Water Inlet Node  →  PACU_VAV_bot CoolC Water Outlet Node
  ✅ PACU_VAV_mid CW Cooling Coil
     Air:   PACU_VAV_mid_OA-PACU_VAV_mid_CoolCNode  →  PACU_VAV_mid_CoolC-PACU_VAV_mid_HeatCNode
     Water: PACU_VAV_mid CoolC Water Inlet Node  →  PACU_VAV_mid CoolC Water Outlet Node
  ✅ PACU_VAV_top CW Cooling Coil
     Air:   PACU_VAV_top_OA-PACU_VAV_top_CoolCNode  →  PACU_VAV_top_CoolC-PACU_VAV_top_HeatCNode
     Water: PACU_VAV_top CoolC Water Inlet Node  →  PACU_VAV_top CoolC Water Outlet Node

  ✅ Coil:Cooling:Water: 6


## Add Coil:Heating:Water (3)

In [17]:
for x in AHU_LIST:
    n = ahu_nodes[x]

    coil = idf.newidfobject("COIL:HEATING:WATER")
    coil.Name                        = n["hw_coil_name"]
    coil.Availability_Schedule_Name  = "ALWAYS_ON"
    coil.UFactor_Times_Area_Value    = "autosize"
    coil.Maximum_Water_Flow_Rate     = "autosize"
    coil.Water_Inlet_Node_Name       = n["heat_water_inlet"]
    coil.Water_Outlet_Node_Name      = n["heat_water_outlet"]
    coil.Air_Inlet_Node_Name         = n["heat_air_inlet"]
    coil.Air_Outlet_Node_Name        = n["heat_air_outlet"]
    coil.Performance_Input_Method    = "UFactorTimesAreaAndDesignWaterFlowRate"
    coil.Rated_Capacity              = "autosize"
    coil.Rated_Inlet_Water_Temperature   = 45.0
    coil.Rated_Inlet_Air_Temperature     = 16.6
    coil.Rated_Outlet_Water_Temperature  = 35.0
    coil.Rated_Outlet_Air_Temperature    = 32.2
    coil.Rated_Ratio_for_Air_and_Water_Convection = 0.5

    print(f"  ✅ {coil.Name}")
    print(f"     Air:   {n['heat_air_inlet']}  →  {n['heat_air_outlet']}")
    print(f"     Water: {n['heat_water_inlet']}  →  {n['heat_water_outlet']}")

print(f"\n  ✅ Coil:Heating:Water: {len(idf.idfobjects['COIL:HEATING:WATER'])}")

  ✅ PACU_VAV_bot HW Heating Coil
     Air:   PACU_VAV_bot_CoolC-PACU_VAV_bot_HeatCNode  →  PACU_VAV_bot_HeatC-PACU_VAV_bot FanNode
     Water: PACU_VAV_bot HeatC Water Inlet Node  →  PACU_VAV_bot HeatC Water Outlet Node
  ✅ PACU_VAV_mid HW Heating Coil
     Air:   PACU_VAV_mid_CoolC-PACU_VAV_mid_HeatCNode  →  PACU_VAV_mid_HeatC-PACU_VAV_mid FanNode
     Water: PACU_VAV_mid HeatC Water Inlet Node  →  PACU_VAV_mid HeatC Water Outlet Node
  ✅ PACU_VAV_top HW Heating Coil
     Air:   PACU_VAV_top_CoolC-PACU_VAV_top_HeatCNode  →  PACU_VAV_top_HeatC-PACU_VAV_top FanNode
     Water: PACU_VAV_top HeatC Water Inlet Node  →  PACU_VAV_top HeatC Water Outlet Node

  ✅ Coil:Heating:Water: 3


## Add Controller:WaterCoil (3)

In [18]:
for x in AHU_LIST:
    n = ahu_nodes[x]

    ctrl = idf.newidfobject("CONTROLLER:WATERCOIL")
    ctrl.Name                            = n["cw_ctrl_name"]
    ctrl.Control_Variable                = "Temperature"
    ctrl.Action                          = "Reverse" 
    ctrl.Actuator_Variable               = "Flow"
    ctrl.Sensor_Node_Name                = n["cool_air_outlet"]
    ctrl.Actuator_Node_Name              = n["cool_water_inlet"]
    ctrl.Controller_Convergence_Tolerance = "autosize"
    ctrl.Maximum_Actuated_Flow           = "autosize"
    ctrl.Minimum_Actuated_Flow           = 0.0

    print(f"  ✅ {ctrl.Name}")
    print(f"     Sensor:   {n['cool_air_outlet']}")
    print(f"     Actuator: {n['cool_water_inlet']}")
    print(f"     Action:   Reverse")

print(f"\n  ✅ Controller:WaterCoil: {len(idf.idfobjects['CONTROLLER:WATERCOIL'])}")

  ✅ PACU_VAV_bot CW CoolC Controller
     Sensor:   PACU_VAV_bot_CoolC-PACU_VAV_bot_HeatCNode
     Actuator: PACU_VAV_bot CoolC Water Inlet Node
     Action:   Reverse
  ✅ PACU_VAV_mid CW CoolC Controller
     Sensor:   PACU_VAV_mid_CoolC-PACU_VAV_mid_HeatCNode
     Actuator: PACU_VAV_mid CoolC Water Inlet Node
     Action:   Reverse
  ✅ PACU_VAV_top CW CoolC Controller
     Sensor:   PACU_VAV_top_CoolC-PACU_VAV_top_HeatCNode
     Actuator: PACU_VAV_top CoolC Water Inlet Node
     Action:   Reverse

  ✅ Controller:WaterCoil: 3


## Add Controller:WaterCoil (3)

In [19]:
for x in AHU_LIST:
    n = ahu_nodes[x]

    ctrl = idf.newidfobject("CONTROLLER:WATERCOIL")
    ctrl.Name                            = n["hw_ctrl_name"]
    ctrl.Control_Variable                = "Temperature"
    ctrl.Action                          = "Normal"
    ctrl.Actuator_Variable               = "Flow"
    ctrl.Sensor_Node_Name                = n["heat_air_outlet"]
    ctrl.Actuator_Node_Name              = n["heat_water_inlet"]
    ctrl.Controller_Convergence_Tolerance = "autosize"
    ctrl.Maximum_Actuated_Flow           = "autosize"
    ctrl.Minimum_Actuated_Flow           = 0.0

    print(f"  ✅ {ctrl.Name}")
    print(f"     Sensor:   {n['heat_air_outlet']}")
    print(f"     Actuator: {n['heat_water_inlet']}")
    print(f"     Action:   Normal")

print(f"\n  ✅ Controller:WaterCoil: {len(idf.idfobjects['CONTROLLER:WATERCOIL'])} 个")

  ✅ PACU_VAV_bot HW HeatC Controller
     Sensor:   PACU_VAV_bot_HeatC-PACU_VAV_bot FanNode
     Actuator: PACU_VAV_bot HeatC Water Inlet Node
     Action:   Normal
  ✅ PACU_VAV_mid HW HeatC Controller
     Sensor:   PACU_VAV_mid_HeatC-PACU_VAV_mid FanNode
     Actuator: PACU_VAV_mid HeatC Water Inlet Node
     Action:   Normal
  ✅ PACU_VAV_top HW HeatC Controller
     Sensor:   PACU_VAV_top_HeatC-PACU_VAV_top FanNode
     Actuator: PACU_VAV_top HeatC Water Inlet Node
     Action:   Normal

  ✅ Controller:WaterCoil: 6 个


## Update Branch

In [20]:
for br in idf.idfobjects["BRANCH"]:
    if "Air Loop Main Branch" not in str(br.Name):
        continue

    for x in AHU_LIST:
        if f"PACU_VAV_{x}" not in str(br.Name):
            continue

        n = ahu_nodes[x]

        # Component 2: CoilSystem:Cooling:DX → Coil:Cooling:Water
        br.Component_2_Object_Type    = "Coil:Cooling:Water"
        br.Component_2_Name           = n["cw_coil_name"]
        br.Component_2_Inlet_Node_Name  = n["cool_air_inlet"]
        br.Component_2_Outlet_Node_Name = n["cool_air_outlet"]

        # Component 3: Coil:Heating:Fuel → Coil:Heating:Water
        br.Component_3_Object_Type    = "Coil:Heating:Water"
        br.Component_3_Name           = n["hw_coil_name"]
        br.Component_3_Inlet_Node_Name  = n["heat_air_inlet"]
        br.Component_3_Outlet_Node_Name = n["heat_air_outlet"]

        print(f"\n  ✅ {br.Name}")
        print(f"     [1] {br.Component_1_Object_Type}: {br.Component_1_Name} (retain)")
        print(f"     [2] {br.Component_2_Object_Type}: {br.Component_2_Name} (new)")
        print(f"         Air: {br.Component_2_Inlet_Node_Name} → {br.Component_2_Outlet_Node_Name}")
        print(f"     [3] {br.Component_3_Object_Type}: {br.Component_3_Name} (new)")
        print(f"         Air: {br.Component_3_Inlet_Node_Name} → {br.Component_3_Outlet_Node_Name}")
        print(f"     [4] {br.Component_4_Object_Type}: {br.Component_4_Name}  (retain)")
        break


  ✅ PACU_VAV_bot Air Loop Main Branch
     [1] AirLoopHVAC:OutdoorAirSystem: PACU_VAV_bot_OA (retain)
     [2] Coil:Cooling:Water: PACU_VAV_bot CW Cooling Coil (new)
         Air: PACU_VAV_bot_OA-PACU_VAV_bot_CoolCNode → PACU_VAV_bot_CoolC-PACU_VAV_bot_HeatCNode
     [3] Coil:Heating:Water: PACU_VAV_bot HW Heating Coil (new)
         Air: PACU_VAV_bot_CoolC-PACU_VAV_bot_HeatCNode → PACU_VAV_bot_HeatC-PACU_VAV_bot FanNode
     [4] Fan:VariableVolume: PACU_VAV_bot Fan  (retain)

  ✅ PACU_VAV_mid Air Loop Main Branch
     [1] AirLoopHVAC:OutdoorAirSystem: PACU_VAV_mid_OA (retain)
     [2] Coil:Cooling:Water: PACU_VAV_mid CW Cooling Coil (new)
         Air: PACU_VAV_mid_OA-PACU_VAV_mid_CoolCNode → PACU_VAV_mid_CoolC-PACU_VAV_mid_HeatCNode
     [3] Coil:Heating:Water: PACU_VAV_mid HW Heating Coil (new)
         Air: PACU_VAV_mid_CoolC-PACU_VAV_mid_HeatCNode → PACU_VAV_mid_HeatC-PACU_VAV_mid FanNode
     [4] Fan:VariableVolume: PACU_VAV_mid Fan  (retain)

  ✅ PACU_VAV_top Air Loop Main Bran

## Create AirLoopHVAC:ControllerList

In [21]:
for x in AHU_LIST:
    n = ahu_nodes[x]

    # Create New ControllerList
    cl = idf.newidfobject("AIRLOOPHVAC:CONTROLLERLIST")
    cl.Name = n["ctrl_list_name"]
    cl.Controller_1_Object_Type = "Controller:WaterCoil"
    cl.Controller_1_Name        = n["cw_ctrl_name"]
    cl.Controller_2_Object_Type = "Controller:WaterCoil"
    cl.Controller_2_Name        = n["hw_ctrl_name"]

    print(f"  ✅ ControllerList: {cl.Name}")
    print(f"     [1] {cl.Controller_1_Object_Type}: {cl.Controller_1_Name}")
    print(f"     [2] {cl.Controller_2_Object_Type}: {cl.Controller_2_Name}")

    # Update AirLoopHVAC
    for airloop in idf.idfobjects["AIRLOOPHVAC"]:
        if str(airloop.Name).strip() == f"PACU_VAV_{x}":
            old_val = airloop.Controller_List_Name
            airloop.Controller_List_Name = n["ctrl_list_name"]
            print(f"     → AirLoopHVAC '{airloop.Name}': Controller_List_Name")
            print(f"       '{old_val}' → '{airloop.Controller_List_Name}'")
            break

print(f"\n  ✅ All AirLoopHVAC:ControllerList:")
for cl in idf.idfobjects["AIRLOOPHVAC:CONTROLLERLIST"]:
    print(f"    • {cl.Name}")

  ✅ ControllerList: PACU_VAV_bot_Water_Coil_Controllers
     [1] Controller:WaterCoil: PACU_VAV_bot CW CoolC Controller
     [2] Controller:WaterCoil: PACU_VAV_bot HW HeatC Controller
     → AirLoopHVAC 'PACU_VAV_bot': Controller_List_Name
       '' → 'PACU_VAV_bot_Water_Coil_Controllers'
  ✅ ControllerList: PACU_VAV_mid_Water_Coil_Controllers
     [1] Controller:WaterCoil: PACU_VAV_mid CW CoolC Controller
     [2] Controller:WaterCoil: PACU_VAV_mid HW HeatC Controller
     → AirLoopHVAC 'PACU_VAV_mid': Controller_List_Name
       '' → 'PACU_VAV_mid_Water_Coil_Controllers'
  ✅ ControllerList: PACU_VAV_top_Water_Coil_Controllers
     [1] Controller:WaterCoil: PACU_VAV_top CW CoolC Controller
     [2] Controller:WaterCoil: PACU_VAV_top HW HeatC Controller
     → AirLoopHVAC 'PACU_VAV_top': Controller_List_Name
       '' → 'PACU_VAV_top_Water_Coil_Controllers'

  ✅ All AirLoopHVAC:ControllerList:
    • PACU_VAV_bot_OA_Controllers
    • PACU_VAV_mid_OA_Controllers
    • PACU_VAV_top_OA_Con

# Phase 3: Create CHW PlantLoop

## Define CHW Loop Node

In [23]:
AHU_LIST = ["bot", "mid", "top"]

CHW = {
    "loop_name":              "CHW Loop",
    "supply_inlet":           "CHW Supply Inlet Node",
    "supply_outlet":          "CHW Supply Outlet Node",
    "demand_inlet":           "CHW Demand Inlet Node",
    "demand_outlet":          "CHW Demand Outlet Node",

    "pump_outlet":            "CHW Pump Outlet Node",
    "equip_hp_inlet":         "GSHP CoolHP Load Inlet Node",
    "equip_hp_outlet":        "GSHP CoolHP Load Outlet Node",
    "supply_bypass_inlet":    "CHW Supply Bypass Inlet Node",
    "supply_bypass_outlet":   "CHW Supply Bypass Outlet Node",
    "supply_mixer_outlet":    "CHW Supply Mixer Outlet Node",

    "demand_pipe_inlet_out":  "CHW Demand Inlet Pipe Outlet Node",
    "demand_bypass_inlet":    "CHW Demand Bypass Inlet Node",
    "demand_bypass_outlet":   "CHW Demand Bypass Outlet Node",
    "demand_mixer_outlet":    "CHW Demand Mixer Outlet Node",

    "hp_source_inlet":        "GSHP CoolHP Source Inlet Node",
    "hp_source_outlet":       "GSHP CoolHP Source Outlet Node",

    "pump_name":              "CHW Pump",
    "hp_cool_name":           "GSHP Cooling HP",
    "hp_heat_companion":      "GSHP Heating HP",
    "spm_name":               "CHW Loop Setpoint Manager",
    "temp_sch_name":          "CHW Loop Temp Schedule",
    "ops_scheme_name":        "CHW Loop Operation Schemes",
    "ops_cooling_name":       "CHW Cooling Operation",
    "equip_list_name":        "CHW Equipment List",
}

CHW_COIL_NODES = {}
for x in AHU_LIST:
    CHW_COIL_NODES[x] = {
        "water_inlet":  f"PACU_VAV_{x} CoolC Water Inlet Node",
        "water_outlet": f"PACU_VAV_{x} CoolC Water Outlet Node",
        "coil_name":    f"PACU_VAV_{x} CW Cooling Coil",
    }

print("=" * 70)
print("  CHW Loop Node and Object Names")
print("=" * 70)
print(f"""
  SUPPLY SIDE:
  {CHW['supply_inlet']}
    → [{CHW['pump_name']}] → {CHW['pump_outlet']}
    → Splitter ─┬─ [{CHW['hp_cool_name']}] ─── {CHW['equip_hp_inlet']} → {CHW['equip_hp_outlet']}
                └─ [Bypass Pipe] ─────── {CHW['supply_bypass_inlet']} → {CHW['supply_bypass_outlet']}
    → Mixer → {CHW['supply_mixer_outlet']}
    → [Outlet Pipe] → {CHW['supply_outlet']}  ← SetpointManager (7°C)

  DEMAND SIDE:
  {CHW['demand_inlet']}
    → [Inlet Pipe] → {CHW['demand_pipe_inlet_out']}
    → Splitter ─┬─ [CW Coil bot] ─── {CHW_COIL_NODES['bot']['water_inlet']} → {CHW_COIL_NODES['bot']['water_outlet']}
                ├─ [CW Coil mid] ─── {CHW_COIL_NODES['mid']['water_inlet']} → {CHW_COIL_NODES['mid']['water_outlet']}
                ├─ [CW Coil top] ─── {CHW_COIL_NODES['top']['water_inlet']} → {CHW_COIL_NODES['top']['water_outlet']}
                └─ [Bypass Pipe] ─── {CHW['demand_bypass_inlet']} → {CHW['demand_bypass_outlet']}
    → Mixer → {CHW['demand_mixer_outlet']}
    → [Outlet Pipe] → {CHW['demand_outlet']}
""")

  CHW Loop Node and Object Names

  SUPPLY SIDE:
  CHW Supply Inlet Node
    → [CHW Pump] → CHW Pump Outlet Node
    → Splitter ─┬─ [GSHP Cooling HP] ─── GSHP CoolHP Load Inlet Node → GSHP CoolHP Load Outlet Node
                └─ [Bypass Pipe] ─────── CHW Supply Bypass Inlet Node → CHW Supply Bypass Outlet Node
    → Mixer → CHW Supply Mixer Outlet Node
    → [Outlet Pipe] → CHW Supply Outlet Node  ← SetpointManager (7°C)

  DEMAND SIDE:
  CHW Demand Inlet Node
    → [Inlet Pipe] → CHW Demand Inlet Pipe Outlet Node
    → Splitter ─┬─ [CW Coil bot] ─── PACU_VAV_bot CoolC Water Inlet Node → PACU_VAV_bot CoolC Water Outlet Node
                ├─ [CW Coil mid] ─── PACU_VAV_mid CoolC Water Inlet Node → PACU_VAV_mid CoolC Water Outlet Node
                ├─ [CW Coil top] ─── PACU_VAV_top CoolC Water Inlet Node → PACU_VAV_top CoolC Water Outlet Node
                └─ [Bypass Pipe] ─── CHW Demand Bypass Inlet Node → CHW Demand Bypass Outlet Node
    → Mixer → CHW Demand Mixer Outlet Node


## Create WWHP Curve:QuadLinear

In [24]:
print("=" * 65)
print("  Create WWHP Performance Curves (Curve:QuadLinear)")
print("=" * 65)

cap_curve = idf.newidfobject("CURVE:QUADLINEAR")
cap_curve.Name           = "GSHP_CoolCap_QuadLinear"
cap_curve.Coefficient1_Constant = -1.52030596
cap_curve.Coefficient2_w = 3.46625667
cap_curve.Coefficient3_x = -1.32267797
cap_curve.Coefficient4_y = 0.09395678
cap_curve.Coefficient5_z = 0.038975504
cap_curve.Minimum_Value_of_w = 0.0
cap_curve.Maximum_Value_of_w = 100.0
cap_curve.Minimum_Value_of_x = 0.0
cap_curve.Maximum_Value_of_x = 100.0
cap_curve.Minimum_Value_of_y = 0.0
cap_curve.Maximum_Value_of_y = 100.0
cap_curve.Minimum_Value_of_z = 0.0
cap_curve.Maximum_Value_of_z = 100.0
print(f"  ✅ {cap_curve.Name}")

pwr_curve = idf.newidfobject("CURVE:QUADLINEAR")
pwr_curve.Name           = "GSHP_CoolPwr_QuadLinear"
pwr_curve.Coefficient1_Constant = -8.59564386
pwr_curve.Coefficient2_w = 0.96265085
pwr_curve.Coefficient3_x = 8.69489229
pwr_curve.Coefficient4_y = 0.02501462
pwr_curve.Coefficient5_z = -0.20132665
pwr_curve.Minimum_Value_of_w = 0.0
pwr_curve.Maximum_Value_of_w = 100.0
pwr_curve.Minimum_Value_of_x = 0.0
pwr_curve.Maximum_Value_of_x = 100.0
pwr_curve.Minimum_Value_of_y = 0.0
pwr_curve.Maximum_Value_of_y = 100.0
pwr_curve.Minimum_Value_of_z = 0.0
pwr_curve.Maximum_Value_of_z = 100.0
print(f"  ✅ {pwr_curve.Name}")

  Create WWHP Performance Curves (Curve:QuadLinear)
  ✅ GSHP_CoolCap_QuadLinear
  ✅ GSHP_CoolPwr_QuadLinear


## Create HeatPump:WaterToWater:EquationFit:Cooling

In [25]:
hp = idf.newidfobject("HEATPUMP:WATERTOWATER:EQUATIONFIT:COOLING")
hp.Name                                = CHW["hp_cool_name"]
hp.Source_Side_Inlet_Node_Name         = CHW["hp_source_inlet"]
hp.Source_Side_Outlet_Node_Name        = CHW["hp_source_outlet"]
hp.Load_Side_Inlet_Node_Name          = CHW["equip_hp_inlet"]
hp.Load_Side_Outlet_Node_Name         = CHW["equip_hp_outlet"]
hp.Companion_Heating_Heat_Pump_Name    = CHW["hp_heat_companion"]
hp.Reference_Load_Side_Flow_Rate      = "autosize"
hp.Reference_Source_Side_Flow_Rate    = "autosize"
hp.Reference_Cooling_Capacity         = "autosize"
hp.Reference_Cooling_Power_Consumption = "autosize"
hp.Cooling_Capacity_Curve_Name        = "GSHP_CoolCap_QuadLinear"
hp.Cooling_Compressor_Power_Curve_Name = "GSHP_CoolPwr_QuadLinear"

print(f"  ✅ {hp.Name}")
print(f"     Load Side:   {hp.Load_Side_Inlet_Node_Name} → {hp.Load_Side_Outlet_Node_Name}  (CHW Supply)")
print(f"     Source Side:  {hp.Source_Side_Inlet_Node_Name} → {hp.Source_Side_Outlet_Node_Name}  (GHE, not modeled in this RP)")
print(f"     Companion:    {hp.Companion_Heating_Heat_Pump_Name}  (HW Loop Phased HP, to be created later)")
print(f"     Capacity/Power: autosize")
print(f"     Curves: {hp.Cooling_Capacity_Curve_Name}, {hp.Cooling_Compressor_Power_Curve_Name}")

  ✅ GSHP Cooling HP
     Load Side:   GSHP CoolHP Load Inlet Node → GSHP CoolHP Load Outlet Node  (CHW Supply)
     Source Side:  GSHP CoolHP Source Inlet Node → GSHP CoolHP Source Outlet Node  (GHE, not modeled in this RP)
     Companion:    GSHP Heating HP  (HW Loop Phased HP, to be created later)
     Capacity/Power: autosize
     Curves: GSHP_CoolCap_QuadLinear, GSHP_CoolPwr_QuadLinear


## Create CHW Supply Side — Pump + Pipes

In [26]:
pump = idf.newidfobject("PUMP:VARIABLESPEED")
pump.Name                              = CHW["pump_name"]
pump.Inlet_Node_Name                   = CHW["supply_inlet"]
pump.Outlet_Node_Name                  = CHW["pump_outlet"]
pump.Design_Maximum_Flow_Rate          = "autosize"
pump.Design_Pump_Head                  = 179352
pump.Design_Power_Consumption          = "autosize"
pump.Motor_Efficiency                  = 0.9
pump.Fraction_of_Motor_Inefficiencies_to_Fluid_Stream = 0.0
pump.Coefficient_1_of_the_Part_Load_Performance_Curve = 0
pump.Coefficient_2_of_the_Part_Load_Performance_Curve = 1
pump.Coefficient_3_of_the_Part_Load_Performance_Curve = 0
pump.Coefficient_4_of_the_Part_Load_Performance_Curve = 0
pump.Design_Minimum_Flow_Rate         = 0.0
pump.Pump_Control_Type                = "Intermittent"
print(f"  ✅ {pump.Name}")
print(f"     {pump.Inlet_Node_Name} → {pump.Outlet_Node_Name}")

pipe_bypass_s = idf.newidfobject("PIPE:ADIABATIC")
pipe_bypass_s.Name           = "CHW Supply Bypass Pipe"
pipe_bypass_s.Inlet_Node_Name  = CHW["supply_bypass_inlet"]
pipe_bypass_s.Outlet_Node_Name = CHW["supply_bypass_outlet"]
print(f"  ✅ {pipe_bypass_s.Name}")

pipe_outlet_s = idf.newidfobject("PIPE:ADIABATIC")
pipe_outlet_s.Name           = "CHW Supply Outlet Pipe"
pipe_outlet_s.Inlet_Node_Name  = CHW["supply_mixer_outlet"]
pipe_outlet_s.Outlet_Node_Name = CHW["supply_outlet"]
print(f"  ✅ {pipe_outlet_s.Name}")

  ✅ CHW Pump
     CHW Supply Inlet Node → CHW Pump Outlet Node
  ✅ CHW Supply Bypass Pipe
  ✅ CHW Supply Outlet Pipe


## Create CHW Supply Side — Branches

In [27]:
supply_branches = [
    ("CHW Supply Inlet Branch",          "Pump:VariableSpeed",
     CHW["pump_name"],                   CHW["supply_inlet"],       CHW["pump_outlet"]),

    ("CHW Supply Equipment Branch",      "HeatPump:WaterToWater:EquationFit:Cooling",
     CHW["hp_cool_name"],                CHW["equip_hp_inlet"],     CHW["equip_hp_outlet"]),

    ("CHW Supply Bypass Branch",         "Pipe:Adiabatic",
     "CHW Supply Bypass Pipe",           CHW["supply_bypass_inlet"], CHW["supply_bypass_outlet"]),

    ("CHW Supply Outlet Branch",         "Pipe:Adiabatic",
     "CHW Supply Outlet Pipe",           CHW["supply_mixer_outlet"], CHW["supply_outlet"]),
]

for br_name, comp_type, comp_name, inlet, outlet in supply_branches:
    br = idf.newidfobject("BRANCH")
    br.Name = br_name
    br.Component_1_Object_Type    = comp_type
    br.Component_1_Name           = comp_name
    br.Component_1_Inlet_Node_Name  = inlet
    br.Component_1_Outlet_Node_Name = outlet
    print(f"  ✅ {br.Name}")
    print(f"     [{comp_type}]: {comp_name}")
    print(f"     {inlet} → {outlet}")


  ✅ CHW Supply Inlet Branch
     [Pump:VariableSpeed]: CHW Pump
     CHW Supply Inlet Node → CHW Pump Outlet Node
  ✅ CHW Supply Equipment Branch
     [HeatPump:WaterToWater:EquationFit:Cooling]: GSHP Cooling HP
     GSHP CoolHP Load Inlet Node → GSHP CoolHP Load Outlet Node
  ✅ CHW Supply Bypass Branch
     [Pipe:Adiabatic]: CHW Supply Bypass Pipe
     CHW Supply Bypass Inlet Node → CHW Supply Bypass Outlet Node
  ✅ CHW Supply Outlet Branch
     [Pipe:Adiabatic]: CHW Supply Outlet Pipe
     CHW Supply Mixer Outlet Node → CHW Supply Outlet Node


## Create CHW Supply Side — BranchList / Splitter / Mixer / ConnectorList

In [28]:
# ── BranchList ──
bl = idf.newidfobject("BRANCHLIST")
bl.Name        = "CHW Supply Branches"
bl.Branch_1_Name = "CHW Supply Inlet Branch"
bl.Branch_2_Name = "CHW Supply Equipment Branch"
bl.Branch_3_Name = "CHW Supply Bypass Branch"
bl.Branch_4_Name = "CHW Supply Outlet Branch"
print(f"  ✅ BranchList: {bl.Name}")

# ── Splitter ──
sp = idf.newidfobject("CONNECTOR:SPLITTER")
sp.Name                     = "CHW Supply Splitter"
sp.Inlet_Branch_Name        = "CHW Supply Inlet Branch"
sp.Outlet_Branch_1_Name     = "CHW Supply Equipment Branch"
sp.Outlet_Branch_2_Name     = "CHW Supply Bypass Branch"
print(f"  ✅ Splitter: {sp.Name}")
print(f"     Inlet:  CHW Supply Inlet Branch")
print(f"     Out 1:  CHW Supply Equipment Branch")
print(f"     Out 2:  CHW Supply Bypass Branch")

# ── Mixer ──
mx = idf.newidfobject("CONNECTOR:MIXER")
mx.Name                    = "CHW Supply Mixer"
mx.Outlet_Branch_Name      = "CHW Supply Outlet Branch"
mx.Inlet_Branch_1_Name     = "CHW Supply Equipment Branch"
mx.Inlet_Branch_2_Name     = "CHW Supply Bypass Branch"
print(f"  ✅ Mixer: {mx.Name}")

# ── ConnectorList ──
cl = idf.newidfobject("CONNECTORLIST")
cl.Name                    = "CHW Supply Connectors"
cl.Connector_1_Object_Type = "Connector:Splitter"
cl.Connector_1_Name        = "CHW Supply Splitter"
cl.Connector_2_Object_Type = "Connector:Mixer"
cl.Connector_2_Name        = "CHW Supply Mixer"
print(f"  ✅ ConnectorList: {cl.Name}")

  ✅ BranchList: CHW Supply Branches
  ✅ Splitter: CHW Supply Splitter
     Inlet:  CHW Supply Inlet Branch
     Out 1:  CHW Supply Equipment Branch
     Out 2:  CHW Supply Bypass Branch
  ✅ Mixer: CHW Supply Mixer
  ✅ ConnectorList: CHW Supply Connectors


## Create CHW Demand Side — Pipes

In [29]:
pipe_inlet_d = idf.newidfobject("PIPE:ADIABATIC")
pipe_inlet_d.Name           = "CHW Demand Inlet Pipe"
pipe_inlet_d.Inlet_Node_Name  = CHW["demand_inlet"]
pipe_inlet_d.Outlet_Node_Name = CHW["demand_pipe_inlet_out"]
print(f"  ✅ {pipe_inlet_d.Name}")

pipe_bypass_d = idf.newidfobject("PIPE:ADIABATIC")
pipe_bypass_d.Name           = "CHW Demand Bypass Pipe"
pipe_bypass_d.Inlet_Node_Name  = CHW["demand_bypass_inlet"]
pipe_bypass_d.Outlet_Node_Name = CHW["demand_bypass_outlet"]
print(f"  ✅ {pipe_bypass_d.Name}")

pipe_outlet_d = idf.newidfobject("PIPE:ADIABATIC")
pipe_outlet_d.Name           = "CHW Demand Outlet Pipe"
pipe_outlet_d.Inlet_Node_Name  = CHW["demand_mixer_outlet"]
pipe_outlet_d.Outlet_Node_Name = CHW["demand_outlet"]
print(f"  ✅ {pipe_outlet_d.Name}")

  ✅ CHW Demand Inlet Pipe
  ✅ CHW Demand Bypass Pipe
  ✅ CHW Demand Outlet Pipe


## Create CHW Demand Side — Branches

In [30]:
# ── Demand Inlet Branch ──
br = idf.newidfobject("BRANCH")
br.Name = "CHW Demand Inlet Branch"
br.Component_1_Object_Type      = "Pipe:Adiabatic"
br.Component_1_Name             = "CHW Demand Inlet Pipe"
br.Component_1_Inlet_Node_Name  = CHW["demand_inlet"]
br.Component_1_Outlet_Node_Name = CHW["demand_pipe_inlet_out"]
print(f"  ✅ {br.Name}")

# ── 3×CoolCoil Demand Branch ──
for x in AHU_LIST:
    cn = CHW_COIL_NODES[x]
    br = idf.newidfobject("BRANCH")
    br.Name = f"CHW Demand {x} CoolCoil Branch"
    br.Component_1_Object_Type      = "Coil:Cooling:Water"
    br.Component_1_Name             = cn["coil_name"]
    br.Component_1_Inlet_Node_Name  = cn["water_inlet"]
    br.Component_1_Outlet_Node_Name = cn["water_outlet"]
    print(f"  ✅ {br.Name}")
    print(f"     [{cn['coil_name']}]: {cn['water_inlet']} → {cn['water_outlet']}")

# ── Demand Bypass Branch ──
br = idf.newidfobject("BRANCH")
br.Name = "CHW Demand Bypass Branch"
br.Component_1_Object_Type      = "Pipe:Adiabatic"
br.Component_1_Name             = "CHW Demand Bypass Pipe"
br.Component_1_Inlet_Node_Name  = CHW["demand_bypass_inlet"]
br.Component_1_Outlet_Node_Name = CHW["demand_bypass_outlet"]
print(f"  ✅ {br.Name}")

# ── Demand Outlet Branch ──
br = idf.newidfobject("BRANCH")
br.Name = "CHW Demand Outlet Branch"
br.Component_1_Object_Type      = "Pipe:Adiabatic"
br.Component_1_Name             = "CHW Demand Outlet Pipe"
br.Component_1_Inlet_Node_Name  = CHW["demand_mixer_outlet"]
br.Component_1_Outlet_Node_Name = CHW["demand_outlet"]
print(f"  ✅ {br.Name}")

  ✅ CHW Demand Inlet Branch
  ✅ CHW Demand bot CoolCoil Branch
     [PACU_VAV_bot CW Cooling Coil]: PACU_VAV_bot CoolC Water Inlet Node → PACU_VAV_bot CoolC Water Outlet Node
  ✅ CHW Demand mid CoolCoil Branch
     [PACU_VAV_mid CW Cooling Coil]: PACU_VAV_mid CoolC Water Inlet Node → PACU_VAV_mid CoolC Water Outlet Node
  ✅ CHW Demand top CoolCoil Branch
     [PACU_VAV_top CW Cooling Coil]: PACU_VAV_top CoolC Water Inlet Node → PACU_VAV_top CoolC Water Outlet Node
  ✅ CHW Demand Bypass Branch
  ✅ CHW Demand Outlet Branch


## Create CHW Demand Side — BranchList / Splitter / Mixer / ConnectorList

In [ ]:
# ── BranchList ──
bl = idf.newidfobject("BRANCHLIST")
bl.Name        = "CHW Demand Branches"
bl.Branch_1_Name = "CHW Demand Inlet Branch"
bl.Branch_2_Name = "CHW Demand bot CoolCoil Branch"
bl.Branch_3_Name = "CHW Demand mid CoolCoil Branch"
bl.Branch_4_Name = "CHW Demand top CoolCoil Branch"
bl.Branch_5_Name = "CHW Demand Bypass Branch"
bl.Branch_6_Name = "CHW Demand Outlet Branch"
print(f"  ✅ BranchList: {bl.Name}")
print(f"     Branches: Inlet → bot/mid/top CoolCoil + Bypass → Outlet")

# ── Splitter ──
sp = idf.newidfobject("CONNECTOR:SPLITTER")
sp.Name                   = "CHW Demand Splitter"
sp.Inlet_Branch_Name      = "CHW Demand Inlet Branch"
sp.Outlet_Branch_1_Name   = "CHW Demand bot CoolCoil Branch"
sp.Outlet_Branch_2_Name   = "CHW Demand mid CoolCoil Branch"
sp.Outlet_Branch_3_Name   = "CHW Demand top CoolCoil Branch"
sp.Outlet_Branch_4_Name   = "CHW Demand Bypass Branch"
print(f"  ✅ Splitter: {sp.Name}")

# ── Mixer ──
mx = idf.newidfobject("CONNECTOR:MIXER")
mx.Name                  = "CHW Demand Mixer"
mx.Outlet_Branch_Name    = "CHW Demand Outlet Branch"
mx.Inlet_Branch_1_Name   = "CHW Demand bot CoolCoil Branch"
mx.Inlet_Branch_2_Name   = "CHW Demand mid CoolCoil Branch"
mx.Inlet_Branch_3_Name   = "CHW Demand top CoolCoil Branch"
mx.Inlet_Branch_4_Name   = "CHW Demand Bypass Branch"
print(f"  ✅ Mixer: {mx.Name}")

# ── ConnectorList ──
cl = idf.newidfobject("CONNECTORLIST")
cl.Name                    = "CHW Demand Connectors"
cl.Connector_1_Object_Type = "Connector:Splitter"
cl.Connector_1_Name        = "CHW Demand Splitter"
cl.Connector_2_Object_Type = "Connector:Mixer"
cl.Connector_2_Name        = "CHW Demand Mixer"
print(f"  ✅ ConnectorList: {cl.Name}")

  ✅ BranchList: CHW Demand Branches
     Branches: Inlet → bot/mid/top CoolCoil + Bypass → Outlet
  ✅ Splitter: CHW Demand Splitter
  ✅ Mixer: CHW Demand Mixer
  ✅ ConnectorList: CHW Demand Connectors


## Create Object PlantLoop

In [32]:
pl = idf.newidfobject("PLANTLOOP")
pl.Name                                = CHW["loop_name"]
pl.Fluid_Type                          = "Water"
pl.Plant_Equipment_Operation_Scheme_Name = CHW["ops_scheme_name"]
pl.Loop_Temperature_Setpoint_Node_Name = CHW["supply_outlet"]
pl.Maximum_Loop_Temperature            = 25.0
pl.Minimum_Loop_Temperature            = 2.0
pl.Maximum_Loop_Flow_Rate              = "autosize"
pl.Minimum_Loop_Flow_Rate              = 0.0
pl.Plant_Loop_Volume                   = "autosize"
pl.Plant_Side_Inlet_Node_Name          = CHW["supply_inlet"]
pl.Plant_Side_Outlet_Node_Name         = CHW["supply_outlet"]
pl.Plant_Side_Branch_List_Name         = "CHW Supply Branches"
pl.Plant_Side_Connector_List_Name      = "CHW Supply Connectors"
pl.Demand_Side_Inlet_Node_Name         = CHW["demand_inlet"]
pl.Demand_Side_Outlet_Node_Name        = CHW["demand_outlet"]
pl.Demand_Side_Branch_List_Name        = "CHW Demand Branches"
pl.Demand_Side_Connector_List_Name     = "CHW Demand Connectors"
pl.Load_Distribution_Scheme            = "Optimal"

print(f"  ✅ {pl.Name}")
print(f"     Supply: {pl.Plant_Side_Inlet_Node_Name} → {pl.Plant_Side_Outlet_Node_Name}")
print(f"     Demand: {pl.Demand_Side_Inlet_Node_Name} → {pl.Demand_Side_Outlet_Node_Name}")
print(f"     Temp Range: {pl.Minimum_Loop_Temperature}°C ~ {pl.Maximum_Loop_Temperature}°C")

  ✅ CHW Loop
     Supply: CHW Supply Inlet Node → CHW Supply Outlet Node
     Demand: CHW Demand Inlet Node → CHW Demand Outlet Node
     Temp Range: 2.0°C ~ 25.0°C


## Sizing:Plant

In [33]:
sp = idf.newidfobject("SIZING:PLANT")
sp.Plant_or_Condenser_Loop_Name = CHW["loop_name"]
sp.Loop_Type                    = "Cooling"
sp.Design_Loop_Exit_Temperature = 7.0
sp.Loop_Design_Temperature_Difference = 5.0

print(f"     Sizing:Plant: {CHW['loop_name']}")
print(f"     Loop Type:    Cooling")
print(f"     Supply Water Temp:     7.0°C")
print(f"     Loop Temperature Difference:   5.0°C")

     Sizing:Plant: CHW Loop
     Loop Type:    Cooling
     Supply Water Temp:     7.0°C
     Loop Temperature Difference:   5.0°C


## PlantEquipmentOperationSchemes

In [34]:
ops = idf.newidfobject("PLANTEQUIPMENTOPERATIONSCHEMES")
ops.Name = CHW["ops_scheme_name"]
ops.Control_Scheme_1_Object_Type  = "PlantEquipmentOperation:CoolingLoad"
ops.Control_Scheme_1_Name         = CHW["ops_cooling_name"]
ops.Control_Scheme_1_Schedule_Name = "ALWAYS_ON"
print(f"  ✅ PlantEquipmentOperationSchemes: {ops.Name}")

op = idf.newidfobject("PLANTEQUIPMENTOPERATION:COOLINGLOAD")
op.Name = CHW["ops_cooling_name"]
op.Load_Range_1_Lower_Limit = 0.0
op.Load_Range_1_Upper_Limit = 1000000000
op.Range_1_Equipment_List_Name = CHW["equip_list_name"]
print(f"  ✅ CoolingLoad Operation: {op.Name}")

el = idf.newidfobject("PLANTEQUIPMENTLIST")
el.Name = CHW["equip_list_name"]
el.Equipment_1_Object_Type = "HeatPump:WaterToWater:EquationFit:Cooling"
el.Equipment_1_Name        = CHW["hp_cool_name"]
print(f"  ✅ PlantEquipmentList: {el.Name}")
print(f"     Equipment: {CHW['hp_cool_name']}")

  ✅ PlantEquipmentOperationSchemes: CHW Loop Operation Schemes
  ✅ CoolingLoad Operation: CHW Cooling Operation
  ✅ PlantEquipmentList: CHW Equipment List
     Equipment: GSHP Cooling HP


## SetpointManager + Supply Hot Water Schedule

In [35]:
# ── CHW Supply Water Temp Schedule ──
sch = idf.newidfobject("SCHEDULE:COMPACT")
sch.Name             = CHW["temp_sch_name"]
sch.Schedule_Type_Limits_Name = "Temperature"
sch.Field_1          = "Through: 12/31"
sch.Field_2          = "For: AllDays"
sch.Field_3          = "Until: 24:00,7.0"
print(f"  ✅ Schedule: {sch.Name}")

# ── SetpointManager:Scheduled ──
spm = idf.newidfobject("SETPOINTMANAGER:SCHEDULED")
spm.Name                        = CHW["spm_name"]
spm.Control_Variable            = "Temperature"
spm.Schedule_Name               = CHW["temp_sch_name"]
spm.Setpoint_Node_or_NodeList_Name = CHW["supply_outlet"]
print(f"  ✅ SetpointManager: {spm.Name}")
print(f"     Node: {CHW['supply_outlet']}")
print(f"     Setpoint: 7°C (via {CHW['temp_sch_name']})")


  ✅ Schedule: CHW Loop Temp Schedule
  ✅ SetpointManager: CHW Loop Setpoint Manager
     Node: CHW Supply Outlet Node
     Setpoint: 7°C (via CHW Loop Temp Schedule)


# Phase 4: Create HW PlantLoop

## Define HW Loop Node

In [36]:
AHU_LIST = ["bot", "mid", "top"]

HW = {
    "loop_name":              "HW Loop",
    "supply_inlet":           "HW Supply Inlet Node",
    "supply_outlet":          "HW Supply Outlet Node",
    "demand_inlet":           "HW Demand Inlet Node",
    "demand_outlet":          "HW Demand Outlet Node",

    "pump_outlet":            "HW Pump Outlet Node",
    "equip_hp_inlet":         "GSHP HeatHP Load Inlet Node",
    "equip_hp_outlet":        "GSHP HeatHP Load Outlet Node",
    "supply_bypass_inlet":    "HW Supply Bypass Inlet Node",
    "supply_bypass_outlet":   "HW Supply Bypass Outlet Node",
    "supply_mixer_outlet":    "HW Supply Mixer Outlet Node",

    "demand_pipe_inlet_out":  "HW Demand Inlet Pipe Outlet Node",
    "demand_bypass_inlet":    "HW Demand Bypass Inlet Node",
    "demand_bypass_outlet":   "HW Demand Bypass Outlet Node",
    "demand_mixer_outlet":    "HW Demand Mixer Outlet Node",

    "hp_source_inlet":        "GSHP HeatHP Source Inlet Node",
    "hp_source_outlet":       "GSHP HeatHP Source Outlet Node",

    "pump_name":              "HW Pump",
    "hp_heat_name":           "GSHP Heating HP",
    "hp_cool_companion":      "GSHP Cooling HP",
    "spm_name":               "HW Loop Setpoint Manager",
    "temp_sch_name":          "HW Loop Temp Schedule",
    "ops_scheme_name":        "HW Loop Operation Schemes",
    "ops_heating_name":       "HW Heating Operation",
    "equip_list_name":        "HW Equipment List",
}

HW_COIL_NODES = {}
for x in AHU_LIST:
    HW_COIL_NODES[x] = {
        "water_inlet":  f"PACU_VAV_{x} HeatC Water Inlet Node",
        "water_outlet": f"PACU_VAV_{x} HeatC Water Outlet Node",
        "coil_name":    f"PACU_VAV_{x} HW Heating Coil",
    }

print("=" * 70)
print("  HW Loop Node and Object Names")
print("=" * 70)
print(f"""
  SUPPLY SIDE:
  {HW['supply_inlet']}
    → [{HW['pump_name']}] → {HW['pump_outlet']}
    → Splitter ─┬─ [{HW['hp_heat_name']}] ─── {HW['equip_hp_inlet']} → {HW['equip_hp_outlet']}
                └─ [Bypass Pipe] ─────── {HW['supply_bypass_inlet']} → {HW['supply_bypass_outlet']}
    → Mixer → {HW['supply_mixer_outlet']}
    → [Outlet Pipe] → {HW['supply_outlet']}  ← SetpointManager (45°C)

  DEMAND SIDE:
  {HW['demand_inlet']}
    → [Inlet Pipe] → {HW['demand_pipe_inlet_out']}
    → Splitter ─┬─ [HW Coil bot] ─── {HW_COIL_NODES['bot']['water_inlet']} → {HW_COIL_NODES['bot']['water_outlet']}
                ├─ [HW Coil mid] ─── {HW_COIL_NODES['mid']['water_inlet']} → {HW_COIL_NODES['mid']['water_outlet']}
                ├─ [HW Coil top] ─── {HW_COIL_NODES['top']['water_inlet']} → {HW_COIL_NODES['top']['water_outlet']}
                └─ [Bypass Pipe] ─── {HW['demand_bypass_inlet']} → {HW['demand_bypass_outlet']}
    → Mixer → {HW['demand_mixer_outlet']}
    → [Outlet Pipe] → {HW['demand_outlet']}
""")


  HW Loop Node and Object Names

  SUPPLY SIDE:
  HW Supply Inlet Node
    → [HW Pump] → HW Pump Outlet Node
    → Splitter ─┬─ [GSHP Heating HP] ─── GSHP HeatHP Load Inlet Node → GSHP HeatHP Load Outlet Node
                └─ [Bypass Pipe] ─────── HW Supply Bypass Inlet Node → HW Supply Bypass Outlet Node
    → Mixer → HW Supply Mixer Outlet Node
    → [Outlet Pipe] → HW Supply Outlet Node  ← SetpointManager (45°C)

  DEMAND SIDE:
  HW Demand Inlet Node
    → [Inlet Pipe] → HW Demand Inlet Pipe Outlet Node
    → Splitter ─┬─ [HW Coil bot] ─── PACU_VAV_bot HeatC Water Inlet Node → PACU_VAV_bot HeatC Water Outlet Node
                ├─ [HW Coil mid] ─── PACU_VAV_mid HeatC Water Inlet Node → PACU_VAV_mid HeatC Water Outlet Node
                ├─ [HW Coil top] ─── PACU_VAV_top HeatC Water Inlet Node → PACU_VAV_top HeatC Water Outlet Node
                └─ [Bypass Pipe] ─── HW Demand Bypass Inlet Node → HW Demand Bypass Outlet Node
    → Mixer → HW Demand Mixer Outlet Node
    → [Outle

## Create WWHP Curve:QuadLinear

In [37]:
cap_curve = idf.newidfobject("CURVE:QUADLINEAR")
cap_curve.Name           = "GSHP_HeatCap_QuadLinear"
cap_curve.Coefficient1_Constant = -3.33491153
cap_curve.Coefficient2_w = -0.51451946
cap_curve.Coefficient3_x = 4.51592706
cap_curve.Coefficient4_y = 0.01797107
cap_curve.Coefficient5_z = 0.155797661
cap_curve.Minimum_Value_of_w = 0.0
cap_curve.Maximum_Value_of_w = 100.0
cap_curve.Minimum_Value_of_x = 0.0
cap_curve.Maximum_Value_of_x = 100.0
cap_curve.Minimum_Value_of_y = 0.0
cap_curve.Maximum_Value_of_y = 100.0
cap_curve.Minimum_Value_of_z = 0.0
cap_curve.Maximum_Value_of_z = 100.0
print(f"  ✅ {cap_curve.Name}")

pwr_curve = idf.newidfobject("CURVE:QUADLINEAR")
pwr_curve.Name           = "GSHP_HeatPwr_QuadLinear"
pwr_curve.Coefficient1_Constant = -8.93121751
pwr_curve.Coefficient2_w = 8.57035762
pwr_curve.Coefficient3_x = 1.29660976
pwr_curve.Coefficient4_y = -0.21629222
pwr_curve.Coefficient5_z = 0.033862378
pwr_curve.Minimum_Value_of_w = 0.0
pwr_curve.Maximum_Value_of_w = 100.0
pwr_curve.Minimum_Value_of_x = 0.0
pwr_curve.Maximum_Value_of_x = 100.0
pwr_curve.Minimum_Value_of_y = 0.0
pwr_curve.Maximum_Value_of_y = 100.0
pwr_curve.Minimum_Value_of_z = 0.0
pwr_curve.Maximum_Value_of_z = 100.0
print(f"  ✅ {pwr_curve.Name}")

  ✅ GSHP_HeatCap_QuadLinear
  ✅ GSHP_HeatPwr_QuadLinear


## Create HeatPump:WaterToWater:EquationFit:Heating

In [38]:
hp = idf.newidfobject("HEATPUMP:WATERTOWATER:EQUATIONFIT:HEATING")
hp.Name                                 = HW["hp_heat_name"]
hp.Source_Side_Inlet_Node_Name          = HW["hp_source_inlet"]
hp.Source_Side_Outlet_Node_Name         = HW["hp_source_outlet"]
hp.Load_Side_Inlet_Node_Name           = HW["equip_hp_inlet"]
hp.Load_Side_Outlet_Node_Name          = HW["equip_hp_outlet"]
hp.Companion_Cooling_Heat_Pump_Name     = HW["hp_cool_companion"]
hp.Reference_Load_Side_Flow_Rate       = "autosize"
hp.Reference_Source_Side_Flow_Rate     = "autosize"
hp.Reference_Heating_Capacity          = "autosize"
hp.Reference_Heating_Power_Consumption = "autosize"
hp.Heating_Capacity_Curve_Name         = "GSHP_HeatCap_QuadLinear"
hp.Heating_Compressor_Power_Curve_Name = "GSHP_HeatPwr_QuadLinear"

print(f"  ✅ {hp.Name}")
print(f"     Load Side:   {hp.Load_Side_Inlet_Node_Name} → {hp.Load_Side_Outlet_Node_Name}  (HW Supply)")
print(f"     Source Side:  {hp.Source_Side_Inlet_Node_Name} → {hp.Source_Side_Outlet_Node_Name}  (→GHE, not modeled in this RP)")
print(f"     Companion:    {hp.Companion_Cooling_Heat_Pump_Name}  (Phase 3 Cooling HP, created above)")

for hp_cool in idf.idfobjects["HEATPUMP:WATERTOWATER:EQUATIONFIT:COOLING"]:
    if str(hp_cool.Companion_Heating_Heat_Pump_Name).strip() == HW["hp_heat_name"]:
        print(f"\n  ✅ Companion Validation:")
        print(f"     {hp_cool.Name} → Companion = {hp_cool.Companion_Heating_Heat_Pump_Name}")
        print(f"     {hp.Name}      → Companion = {hp.Companion_Cooling_Heat_Pump_Name}")


  ✅ GSHP Heating HP
     Load Side:   GSHP HeatHP Load Inlet Node → GSHP HeatHP Load Outlet Node  (HW Supply)
     Source Side:  GSHP HeatHP Source Inlet Node → GSHP HeatHP Source Outlet Node  (→GHE, not modeled in this RP)
     Companion:    GSHP Cooling HP  (Phase 3 Cooling HP, created above)

  ✅ Companion Validation:
     GSHP Cooling HP → Companion = GSHP Heating HP
     GSHP Heating HP      → Companion = GSHP Cooling HP


## Create HW Supply Side — Pump + Pipes

In [39]:
pump = idf.newidfobject("PUMP:VARIABLESPEED")
pump.Name                              = HW["pump_name"]
pump.Inlet_Node_Name                   = HW["supply_inlet"]
pump.Outlet_Node_Name                  = HW["pump_outlet"]
pump.Design_Maximum_Flow_Rate          = "autosize"
pump.Design_Pump_Head                  = 179352   # Pa (~60 ft H2O)
pump.Design_Power_Consumption          = "autosize"
pump.Motor_Efficiency                  = 0.9
pump.Fraction_of_Motor_Inefficiencies_to_Fluid_Stream = 0.0
pump.Coefficient_1_of_the_Part_Load_Performance_Curve = 0
pump.Coefficient_2_of_the_Part_Load_Performance_Curve = 1
pump.Coefficient_3_of_the_Part_Load_Performance_Curve = 0
pump.Coefficient_4_of_the_Part_Load_Performance_Curve = 0
pump.Design_Minimum_Flow_Rate         = 0.0
pump.Pump_Control_Type                = "Intermittent"
print(f"  ✅ {pump.Name}")

# ── Supply Pipe ──
pipe = idf.newidfobject("PIPE:ADIABATIC")
pipe.Name             = "HW Supply Bypass Pipe"
pipe.Inlet_Node_Name  = HW["supply_bypass_inlet"]
pipe.Outlet_Node_Name = HW["supply_bypass_outlet"]
print(f"  ✅ {pipe.Name}")

# ── Supply Outlet Pipe ──
pipe = idf.newidfobject("PIPE:ADIABATIC")
pipe.Name             = "HW Supply Outlet Pipe"
pipe.Inlet_Node_Name  = HW["supply_mixer_outlet"]
pipe.Outlet_Node_Name = HW["supply_outlet"]
print(f"  ✅ {pipe.Name}")


  ✅ HW Pump
  ✅ HW Supply Bypass Pipe
  ✅ HW Supply Outlet Pipe


## Create HW Supply Side — Branches

In [40]:
supply_branches = [
    ("HW Supply Inlet Branch",     "Pump:VariableSpeed",
     HW["pump_name"],              HW["supply_inlet"],          HW["pump_outlet"]),

    ("HW Supply Equipment Branch", "HeatPump:WaterToWater:EquationFit:Heating",
     HW["hp_heat_name"],           HW["equip_hp_inlet"],        HW["equip_hp_outlet"]),

    ("HW Supply Bypass Branch",    "Pipe:Adiabatic",
     "HW Supply Bypass Pipe",      HW["supply_bypass_inlet"],   HW["supply_bypass_outlet"]),

    ("HW Supply Outlet Branch",    "Pipe:Adiabatic",
     "HW Supply Outlet Pipe",      HW["supply_mixer_outlet"],   HW["supply_outlet"]),
]

for br_name, comp_type, comp_name, inlet, outlet in supply_branches:
    br = idf.newidfobject("BRANCH")
    br.Name = br_name
    br.Component_1_Object_Type      = comp_type
    br.Component_1_Name             = comp_name
    br.Component_1_Inlet_Node_Name  = inlet
    br.Component_1_Outlet_Node_Name = outlet
    print(f"  ✅ {br.Name}")
    print(f"     [{comp_type}]: {inlet} → {outlet}")

  ✅ HW Supply Inlet Branch
     [Pump:VariableSpeed]: HW Supply Inlet Node → HW Pump Outlet Node
  ✅ HW Supply Equipment Branch
     [HeatPump:WaterToWater:EquationFit:Heating]: GSHP HeatHP Load Inlet Node → GSHP HeatHP Load Outlet Node
  ✅ HW Supply Bypass Branch
     [Pipe:Adiabatic]: HW Supply Bypass Inlet Node → HW Supply Bypass Outlet Node
  ✅ HW Supply Outlet Branch
     [Pipe:Adiabatic]: HW Supply Mixer Outlet Node → HW Supply Outlet Node


## Create HW Supply Side — BranchList / Splitter / Mixer / ConnectorList

In [41]:
bl = idf.newidfobject("BRANCHLIST")
bl.Name          = "HW Supply Branches"
bl.Branch_1_Name = "HW Supply Inlet Branch"
bl.Branch_2_Name = "HW Supply Equipment Branch"
bl.Branch_3_Name = "HW Supply Bypass Branch"
bl.Branch_4_Name = "HW Supply Outlet Branch"
print(f"  ✅ BranchList: {bl.Name}")

sp = idf.newidfobject("CONNECTOR:SPLITTER")
sp.Name                   = "HW Supply Splitter"
sp.Inlet_Branch_Name      = "HW Supply Inlet Branch"
sp.Outlet_Branch_1_Name   = "HW Supply Equipment Branch"
sp.Outlet_Branch_2_Name   = "HW Supply Bypass Branch"
print(f"  ✅ Splitter: {sp.Name}")

mx = idf.newidfobject("CONNECTOR:MIXER")
mx.Name                  = "HW Supply Mixer"
mx.Outlet_Branch_Name    = "HW Supply Outlet Branch"
mx.Inlet_Branch_1_Name   = "HW Supply Equipment Branch"
mx.Inlet_Branch_2_Name   = "HW Supply Bypass Branch"
print(f"  ✅ Mixer: {mx.Name}")

cl = idf.newidfobject("CONNECTORLIST")
cl.Name                    = "HW Supply Connectors"
cl.Connector_1_Object_Type = "Connector:Splitter"
cl.Connector_1_Name        = "HW Supply Splitter"
cl.Connector_2_Object_Type = "Connector:Mixer"
cl.Connector_2_Name        = "HW Supply Mixer"
print(f"  ✅ ConnectorList: {cl.Name}")


  ✅ BranchList: HW Supply Branches
  ✅ Splitter: HW Supply Splitter
  ✅ Mixer: HW Supply Mixer
  ✅ ConnectorList: HW Supply Connectors


## Create HW Demand Side — Pipes 

In [42]:
pipe = idf.newidfobject("PIPE:ADIABATIC")
pipe.Name             = "HW Demand Inlet Pipe"
pipe.Inlet_Node_Name  = HW["demand_inlet"]
pipe.Outlet_Node_Name = HW["demand_pipe_inlet_out"]
print(f"  ✅ {pipe.Name}")

pipe = idf.newidfobject("PIPE:ADIABATIC")
pipe.Name             = "HW Demand Bypass Pipe"
pipe.Inlet_Node_Name  = HW["demand_bypass_inlet"]
pipe.Outlet_Node_Name = HW["demand_bypass_outlet"]
print(f"  ✅ {pipe.Name}")

pipe = idf.newidfobject("PIPE:ADIABATIC")
pipe.Name             = "HW Demand Outlet Pipe"
pipe.Inlet_Node_Name  = HW["demand_mixer_outlet"]
pipe.Outlet_Node_Name = HW["demand_outlet"]
print(f"  ✅ {pipe.Name}")

  ✅ HW Demand Inlet Pipe
  ✅ HW Demand Bypass Pipe
  ✅ HW Demand Outlet Pipe


## Create HW Demand Side — Branches

In [43]:
# ── Demand Inlet Branch ──
br = idf.newidfobject("BRANCH")
br.Name = "HW Demand Inlet Branch"
br.Component_1_Object_Type      = "Pipe:Adiabatic"
br.Component_1_Name             = "HW Demand Inlet Pipe"
br.Component_1_Inlet_Node_Name  = HW["demand_inlet"]
br.Component_1_Outlet_Node_Name = HW["demand_pipe_inlet_out"]
print(f"  ✅ {br.Name}")

# ── 3×HeatCoil Demand Branch ──
for x in AHU_LIST:
    cn = HW_COIL_NODES[x]
    br = idf.newidfobject("BRANCH")
    br.Name = f"HW Demand {x} HeatCoil Branch"
    br.Component_1_Object_Type      = "Coil:Heating:Water"
    br.Component_1_Name             = cn["coil_name"]
    br.Component_1_Inlet_Node_Name  = cn["water_inlet"]
    br.Component_1_Outlet_Node_Name = cn["water_outlet"]
    print(f"  ✅ {br.Name}")
    print(f"     [{cn['coil_name']}]: {cn['water_inlet']} → {cn['water_outlet']}")

# ── Demand Bypass Branch ──
br = idf.newidfobject("BRANCH")
br.Name = "HW Demand Bypass Branch"
br.Component_1_Object_Type      = "Pipe:Adiabatic"
br.Component_1_Name             = "HW Demand Bypass Pipe"
br.Component_1_Inlet_Node_Name  = HW["demand_bypass_inlet"]
br.Component_1_Outlet_Node_Name = HW["demand_bypass_outlet"]
print(f"  ✅ {br.Name}")

# ── Demand Outlet Branch ──
br = idf.newidfobject("BRANCH")
br.Name = "HW Demand Outlet Branch"
br.Component_1_Object_Type      = "Pipe:Adiabatic"
br.Component_1_Name             = "HW Demand Outlet Pipe"
br.Component_1_Inlet_Node_Name  = HW["demand_mixer_outlet"]
br.Component_1_Outlet_Node_Name = HW["demand_outlet"]
print(f"  ✅ {br.Name}")

  ✅ HW Demand Inlet Branch
  ✅ HW Demand bot HeatCoil Branch
     [PACU_VAV_bot HW Heating Coil]: PACU_VAV_bot HeatC Water Inlet Node → PACU_VAV_bot HeatC Water Outlet Node
  ✅ HW Demand mid HeatCoil Branch
     [PACU_VAV_mid HW Heating Coil]: PACU_VAV_mid HeatC Water Inlet Node → PACU_VAV_mid HeatC Water Outlet Node
  ✅ HW Demand top HeatCoil Branch
     [PACU_VAV_top HW Heating Coil]: PACU_VAV_top HeatC Water Inlet Node → PACU_VAV_top HeatC Water Outlet Node
  ✅ HW Demand Bypass Branch
  ✅ HW Demand Outlet Branch


## Create HW Demand Side — BranchList / Splitter / Mixer / ConnectorList

In [ ]:
bl = idf.newidfobject("BRANCHLIST")
bl.Name          = "HW Demand Branches"
bl.Branch_1_Name = "HW Demand Inlet Branch"
bl.Branch_2_Name = "HW Demand bot HeatCoil Branch"
bl.Branch_3_Name = "HW Demand mid HeatCoil Branch"
bl.Branch_4_Name = "HW Demand top HeatCoil Branch"
bl.Branch_5_Name = "HW Demand Bypass Branch"
bl.Branch_6_Name = "HW Demand Outlet Branch"
print(f"  ✅ BranchList: {bl.Name}")

sp = idf.newidfobject("CONNECTOR:SPLITTER")
sp.Name                   = "HW Demand Splitter"
sp.Inlet_Branch_Name      = "HW Demand Inlet Branch"
sp.Outlet_Branch_1_Name   = "HW Demand bot HeatCoil Branch"
sp.Outlet_Branch_2_Name   = "HW Demand mid HeatCoil Branch"
sp.Outlet_Branch_3_Name   = "HW Demand top HeatCoil Branch"
sp.Outlet_Branch_4_Name   = "HW Demand Bypass Branch"
print(f"  ✅ Splitter: {sp.Name}")

mx = idf.newidfobject("CONNECTOR:MIXER")
mx.Name                  = "HW Demand Mixer"
mx.Outlet_Branch_Name    = "HW Demand Outlet Branch"
mx.Inlet_Branch_1_Name   = "HW Demand bot HeatCoil Branch"
mx.Inlet_Branch_2_Name   = "HW Demand mid HeatCoil Branch"
mx.Inlet_Branch_3_Name   = "HW Demand top HeatCoil Branch"
mx.Inlet_Branch_4_Name   = "HW Demand Bypass Branch"
print(f"  ✅ Mixer: {mx.Name}")

cl = idf.newidfobject("CONNECTORLIST")
cl.Name                    = "HW Demand Connectors"
cl.Connector_1_Object_Type = "Connector:Splitter"
cl.Connector_1_Name        = "HW Demand Splitter"
cl.Connector_2_Object_Type = "Connector:Mixer"
cl.Connector_2_Name        = "HW Demand Mixer"
print(f"  ✅ ConnectorList: {cl.Name}")

  ✅ BranchList: HW Demand Branches
  ✅ Splitter: HW Demand Splitter
  ✅ Mixer: HW Demand Mixer
  ✅ ConnectorList: HW Demand Connectors


## Create Object PlantLoop

In [45]:
pl = idf.newidfobject("PLANTLOOP")
pl.Name                                = HW["loop_name"]
pl.Fluid_Type                          = "Water"
pl.Plant_Equipment_Operation_Scheme_Name = HW["ops_scheme_name"]
pl.Loop_Temperature_Setpoint_Node_Name = HW["supply_outlet"]
pl.Maximum_Loop_Temperature            = 60.0
pl.Minimum_Loop_Temperature            = 10.0
pl.Maximum_Loop_Flow_Rate              = "autosize"
pl.Minimum_Loop_Flow_Rate              = 0.0
pl.Plant_Loop_Volume                   = "autosize"
pl.Plant_Side_Inlet_Node_Name          = HW["supply_inlet"]
pl.Plant_Side_Outlet_Node_Name         = HW["supply_outlet"]
pl.Plant_Side_Branch_List_Name         = "HW Supply Branches"
pl.Plant_Side_Connector_List_Name      = "HW Supply Connectors"
pl.Demand_Side_Inlet_Node_Name         = HW["demand_inlet"]
pl.Demand_Side_Outlet_Node_Name        = HW["demand_outlet"]
pl.Demand_Side_Branch_List_Name        = "HW Demand Branches"
pl.Demand_Side_Connector_List_Name     = "HW Demand Connectors"
pl.Load_Distribution_Scheme            = "Optimal"

print(f"  ✅ {pl.Name}")
print(f"     Supply: {pl.Plant_Side_Inlet_Node_Name} → {pl.Plant_Side_Outlet_Node_Name}")
print(f"     Demand: {pl.Demand_Side_Inlet_Node_Name} → {pl.Demand_Side_Outlet_Node_Name}")
print(f"     Temp Range: {pl.Minimum_Loop_Temperature}°C ~ {pl.Maximum_Loop_Temperature}°C")


  ✅ HW Loop
     Supply: HW Supply Inlet Node → HW Supply Outlet Node
     Demand: HW Demand Inlet Node → HW Demand Outlet Node
     Temp Range: 10.0°C ~ 60.0°C


## Sizing:Plant

In [46]:
sp = idf.newidfobject("SIZING:PLANT")
sp.Plant_or_Condenser_Loop_Name = HW["loop_name"]
sp.Loop_Type                    = "Heating"
sp.Design_Loop_Exit_Temperature = 45.0
sp.Loop_Design_Temperature_Difference = 11.0

print(f"     Sizing:Plant: {HW['loop_name']}")
print(f"     Loop Type:    Heating")
print(f"     Supply Temperature:     45.0°C")
print(f"     Temperature Difference:   11.0°C")

     Sizing:Plant: HW Loop
     Loop Type:    Heating
     Supply Temperature:     45.0°C
     Temperature Difference:   11.0°C


## PlantEquipmentOperationSchemes

In [47]:
ops = idf.newidfobject("PLANTEQUIPMENTOPERATIONSCHEMES")
ops.Name = HW["ops_scheme_name"]
ops.Control_Scheme_1_Object_Type   = "PlantEquipmentOperation:HeatingLoad"
ops.Control_Scheme_1_Name          = HW["ops_heating_name"]
ops.Control_Scheme_1_Schedule_Name = "ALWAYS_ON"
print(f"  ✅ PlantEquipmentOperationSchemes: {ops.Name}")

op = idf.newidfobject("PLANTEQUIPMENTOPERATION:HEATINGLOAD")
op.Name = HW["ops_heating_name"]
op.Load_Range_1_Lower_Limit    = 0.0
op.Load_Range_1_Upper_Limit    = 1000000000
op.Range_1_Equipment_List_Name = HW["equip_list_name"]
print(f"  ✅ HeatingLoad Operation: {op.Name}")

el = idf.newidfobject("PLANTEQUIPMENTLIST")
el.Name = HW["equip_list_name"]
el.Equipment_1_Object_Type = "HeatPump:WaterToWater:EquationFit:Heating"
el.Equipment_1_Name        = HW["hp_heat_name"]
print(f"  ✅ PlantEquipmentList: {el.Name}")
print(f"     Equipment: {HW['hp_heat_name']}")


  ✅ PlantEquipmentOperationSchemes: HW Loop Operation Schemes
  ✅ HeatingLoad Operation: HW Heating Operation
  ✅ PlantEquipmentList: HW Equipment List
     Equipment: GSHP Heating HP


## SetpointManager + Supply Hot Water Schedule

In [48]:
sch = idf.newidfobject("SCHEDULE:COMPACT")
sch.Name             = HW["temp_sch_name"]
sch.Schedule_Type_Limits_Name = "Temperature"
sch.Field_1          = "Through: 12/31"
sch.Field_2          = "For: AllDays"
sch.Field_3          = "Until: 24:00,45.0"
print(f"  ✅ Schedule: {sch.Name}")

spm = idf.newidfobject("SETPOINTMANAGER:SCHEDULED")
spm.Name                        = HW["spm_name"]
spm.Control_Variable            = "Temperature"
spm.Schedule_Name               = HW["temp_sch_name"]
spm.Setpoint_Node_or_NodeList_Name = HW["supply_outlet"]
print(f"  ✅ SetpointManager: {spm.Name}")
print(f"     Node: {HW['supply_outlet']}")
print(f"     Setpoint: 45°C (via {HW['temp_sch_name']})")

  ✅ Schedule: HW Loop Temp Schedule
  ✅ SetpointManager: HW Loop Setpoint Manager
     Node: HW Supply Outlet Node
     Setpoint: 45°C (via HW Loop Temp Schedule)


# Phase 5: Create GHE (Ground Loop)

## Define GHE Loop Node

In [49]:
GHE = {
    "loop_name":              "GHE Loop",
    "supply_inlet":           "GHE Supply Inlet Node",
    "supply_outlet":          "GHE Supply Outlet Node",
    "demand_inlet":           "GHE Demand Inlet Node",
    "demand_outlet":          "GHE Demand Outlet Node",

    "pump_outlet":            "GHE Pump Outlet Node",
    "ghe_inlet":              "GHE HX Inlet Node",
    "ghe_outlet":             "GHE HX Outlet Node",
    "supply_bypass_inlet":    "GHE Supply Bypass Inlet Node",
    "supply_bypass_outlet":   "GHE Supply Bypass Outlet Node",
    "supply_mixer_outlet":    "GHE Supply Mixer Outlet Node",

    "demand_pipe_inlet_out":  "GHE Demand Inlet Pipe Outlet Node",
    "demand_bypass_inlet":    "GHE Demand Bypass Inlet Node",
    "demand_bypass_outlet":   "GHE Demand Bypass Outlet Node",
    "demand_mixer_outlet":    "GHE Demand Mixer Outlet Node",

    "cool_hp_source_inlet":   "GSHP CoolHP Source Inlet Node",
    "cool_hp_source_outlet":  "GSHP CoolHP Source Outlet Node",
    "heat_hp_source_inlet":   "GSHP HeatHP Source Inlet Node",
    "heat_hp_source_outlet":  "GSHP HeatHP Source Outlet Node",

    "pump_name":              "GHE Pump",
    "ghe_name":               "Vertical GHE System",
    "ghe_props_name":         "GHE Borehole Properties",
    "ground_temp_name":       "NYC Undisturbed Ground Temp",
    "spm_name":               "GHE Loop Setpoint Manager",
    "temp_sch_name":          "GHE Loop Temp Schedule",
    "ops_scheme_name":        "GHE Loop Operation Schemes",
    "ops_uncontrolled_name":  "GHE Uncontrolled Operation",
    "equip_list_name":        "GHE Equipment List",
}

print("=" * 70)
print("  GHE Loop Node and Object Names")
print("=" * 70)
print(f"""
  SUPPLY SIDE:
  {GHE['supply_inlet']}
    → [{GHE['pump_name']}] → {GHE['pump_outlet']}
    → Splitter ─┬─ [{GHE['ghe_name']}] ─ {GHE['ghe_inlet']} → {GHE['ghe_outlet']}
                └─ [Bypass Pipe] ────── {GHE['supply_bypass_inlet']} → {GHE['supply_bypass_outlet']}
    → Mixer → {GHE['supply_mixer_outlet']}
    → [Outlet Pipe] → {GHE['supply_outlet']}

  DEMAND SIDE:
  {GHE['demand_inlet']}
    → [Inlet Pipe] → {GHE['demand_pipe_inlet_out']}
    → Splitter ─┬─ [GSHP Cooling HP Source] ─ {GHE['cool_hp_source_inlet']} → {GHE['cool_hp_source_outlet']}
                ├─ [GSHP Heating HP Source] ─ {GHE['heat_hp_source_inlet']} → {GHE['heat_hp_source_outlet']}
                └─ [Bypass Pipe] ─────────── {GHE['demand_bypass_inlet']} → {GHE['demand_bypass_outlet']}
    → Mixer → {GHE['demand_mixer_outlet']}
    → [Outlet Pipe] → {GHE['demand_outlet']}
""")

  GHE Loop Node and Object Names

  SUPPLY SIDE:
  GHE Supply Inlet Node
    → [GHE Pump] → GHE Pump Outlet Node
    → Splitter ─┬─ [Vertical GHE System] ─ GHE HX Inlet Node → GHE HX Outlet Node
                └─ [Bypass Pipe] ────── GHE Supply Bypass Inlet Node → GHE Supply Bypass Outlet Node
    → Mixer → GHE Supply Mixer Outlet Node
    → [Outlet Pipe] → GHE Supply Outlet Node

  DEMAND SIDE:
  GHE Demand Inlet Node
    → [Inlet Pipe] → GHE Demand Inlet Pipe Outlet Node
    → Splitter ─┬─ [GSHP Cooling HP Source] ─ GSHP CoolHP Source Inlet Node → GSHP CoolHP Source Outlet Node
                ├─ [GSHP Heating HP Source] ─ GSHP HeatHP Source Inlet Node → GSHP HeatHP Source Outlet Node
                └─ [Bypass Pipe] ─────────── GHE Demand Bypass Inlet Node → GHE Demand Bypass Outlet Node
    → Mixer → GHE Demand Mixer Outlet Node
    → [Outlet Pipe] → GHE Demand Outlet Node



## GHE fieldname test

In [50]:
# ── GroundHeatExchanger:System ──
try:
    tmp = idf.newidfobject("GROUNDHEATEXCHANGER:SYSTEM")
    print(f"\n[GroundHeatExchanger:System] fieldnames:")
    for i, fn in enumerate(tmp.fieldnames):
        print(f"  [{i:2d}] {fn}")
    idf.removeidfobject(tmp)
except Exception as e:
    print(f"  ⚠️  Fail: {e}")
    print(f"  → Confirm IDD Objects")

# ── GroundHeatExchanger:Vertical:Properties ──
try:
    tmp = idf.newidfobject("GROUNDHEATEXCHANGER:VERTICAL:PROPERTIES")
    print(f"\n[GroundHeatExchanger:Vertical:Properties] fieldnames:")
    for i, fn in enumerate(tmp.fieldnames):
        print(f"  [{i:2d}] {fn}")
    idf.removeidfobject(tmp)
except Exception as e:
    print(f"  ⚠️  Fail: {e}")

# ── Site:GroundTemperature:Undisturbed:KusudaAchenbach ──
try:
    tmp = idf.newidfobject("SITE:GROUNDTEMPERATURE:UNDISTURBED:KUSUDAACHENBACH")
    print(f"\n[Site:GroundTemperature:Undisturbed:KusudaAchenbach] fieldnames:")
    for i, fn in enumerate(tmp.fieldnames):
        print(f"  [{i:2d}] {fn}")
    idf.removeidfobject(tmp)
except Exception as e:
    print(f"  ⚠️  Fail: {e}")



[GroundHeatExchanger:System] fieldnames:
  [ 0] key
  [ 1] Name
  [ 2] Inlet_Node_Name
  [ 3] Outlet_Node_Name
  [ 4] Design_Flow_Rate
  [ 5] Undisturbed_Ground_Temperature_Model_Type
  [ 6] Undisturbed_Ground_Temperature_Model_Name
  [ 7] Ground_Thermal_Conductivity
  [ 8] Ground_Thermal_Heat_Capacity
  [ 9] GHEVerticalResponseFactors_Object_Name
  [10] gFunction_Calculation_Method
  [11] GHEVerticalArray_Object_Name
  [12] GHEVerticalSingle_Object_Name_1
  [13] GHEVerticalSingle_Object_Name_2
  [14] GHEVerticalSingle_Object_Name_3
  [15] GHEVerticalSingle_Object_Name_4
  [16] GHEVerticalSingle_Object_Name_5
  [17] GHEVerticalSingle_Object_Name_6
  [18] GHEVerticalSingle_Object_Name_7
  [19] GHEVerticalSingle_Object_Name_8
  [20] GHEVerticalSingle_Object_Name_9
  [21] GHEVerticalSingle_Object_Name_10
  [22] GHEVerticalSingle_Object_Name_11
  [23] GHEVerticalSingle_Object_Name_12
  [24] GHEVerticalSingle_Object_Name_13
  [25] GHEVerticalSingle_Object_Name_14
  [26] GHEVerticalSingle_O

## Create Site:GroundTemperature:Undisturbed:KusudaAchenbach

In [51]:
gt = idf.newidfobject("SITE:GROUNDTEMPERATURE:UNDISTURBED:KUSUDAACHENBACH")
gt.Name                                    = GHE["ground_temp_name"]
gt.Soil_Thermal_Conductivity               = 1.8
gt.Soil_Density                            = 2500.0
gt.Soil_Specific_Heat                      = 880.0
gt.Average_Soil_Surface_Temperature        = 12.1
gt.Average_Amplitude_of_Surface_Temperature = 12.3
gt.Phase_Shift_of_Minimum_Surface_Temperature = 15.0

print(f"  ✅ {gt.Name}")
print(f"     Thermal Conductivity: {gt.Soil_Thermal_Conductivity} W/(m·K)")
print(f"     Average Temp: {gt.Average_Soil_Surface_Temperature}°C")
print(f"     Average Amplitude:   {gt.Average_Amplitude_of_Surface_Temperature}°C")


  ✅ NYC Undisturbed Ground Temp
     Thermal Conductivity: 1.8 W/(m·K)
     Average Temp: 12.1°C
     Average Amplitude:   12.3°C


## Create GroundHeatExchanger:Vertical:Properties

In [52]:
props = idf.newidfobject("GROUNDHEATEXCHANGER:VERTICAL:PROPERTIES")
props.Name                       = GHE["ghe_props_name"]
props.Depth_of_Top_of_Borehole   = 1.0
props.Borehole_Length             = 76.2
props.Borehole_Diameter           = 0.1143
props.Grout_Thermal_Conductivity  = 1.0
props.Grout_Thermal_Heat_Capacity = 3.9e6
props.Pipe_Thermal_Conductivity   = 0.4
props.Pipe_Thermal_Heat_Capacity  = 1.542e6
props.Pipe_Outer_Diameter         = 0.0267
props.UTube_Distance              = 0.0521
props.Pipe_Thickness              = 0.00243

print(f"  ✅ {props.Name}")
print(f"     Borehole Length: {props.Borehole_Length} m")
print(f"     Borehole Diameter: {props.Borehole_Diameter} m")
print(f"     Grout Thermal Conductivity: {props.Grout_Thermal_Conductivity} W/(m·K)")
print(f"     Pipe Material: HDPE SDR-11, OD={props.Pipe_Outer_Diameter}m, Cp={props.Pipe_Thermal_Heat_Capacity:.3e} J/(m³·K)")


  ✅ GHE Borehole Properties
     Borehole Length: 76.2 m
     Borehole Diameter: 0.1143 m
     Grout Thermal Conductivity: 1.0 W/(m·K)
     Pipe Material: HDPE SDR-11, OD=0.0267m, Cp=1.542e+06 J/(m³·K)


## Create GHE:Vertical:Array

In [53]:
arr = idf.newidfobject("GROUNDHEATEXCHANGER:VERTICAL:ARRAY")
arr.Name                                   = "GHE Borehole Array"
arr.GHEVerticalProperties_Object_Name      = "GHE Borehole Properties"
arr.Number_of_Boreholes_in_XDirection      = 6
arr.Number_of_Boreholes_in_YDirection      = 6
arr.Borehole_Spacing                       = 6.0

print(f"  ✅ {arr.Name}")
print(f"     Properties: {arr.GHEVerticalProperties_Object_Name}")
print(f"     Boreholes: {arr.Number_of_Boreholes_in_XDirection} × {arr.Number_of_Boreholes_in_YDirection}"
      f" = {arr.Number_of_Boreholes_in_XDirection * arr.Number_of_Boreholes_in_YDirection} holes")
print(f"     Spacing: {arr.Borehole_Spacing} m")
print(f"     Depth: 76.2 m (already set in GHE:Vertical:Properties)")
print(f"     Total Heat Exchanger Length: {arr.Number_of_Boreholes_in_XDirection * arr.Number_of_Boreholes_in_YDirection * 76.2:.0f} m")


  ✅ GHE Borehole Array
     Properties: GHE Borehole Properties
     Boreholes: 6 × 6 = 36 holes
     Spacing: 6.0 m
     Depth: 76.2 m (already set in GHE:Vertical:Properties)
     Total Heat Exchanger Length: 2743 m


## Create GroundHeatExchanger:System


In [54]:
ghe = idf.newidfobject("GROUNDHEATEXCHANGER:SYSTEM")
ghe.Name                                       = GHE["ghe_name"]
ghe.Inlet_Node_Name                            = GHE["ghe_inlet"]
ghe.Outlet_Node_Name                           = GHE["ghe_outlet"]
ghe.Design_Flow_Rate                           = 0.01344
ghe.Undisturbed_Ground_Temperature_Model_Type  = "Site:GroundTemperature:Undisturbed:KusudaAchenbach"
ghe.Undisturbed_Ground_Temperature_Model_Name  = GHE["ground_temp_name"]
ghe.Ground_Thermal_Conductivity                = 1.8
ghe.Ground_Thermal_Heat_Capacity               = 2.2e6
ghe.GHEVerticalResponseFactors_Object_Name     = "GHE Response Factors"

print(f"  ✅ {ghe.Name}")
print(f"     Nodes: {ghe.Inlet_Node_Name} → {ghe.Outlet_Node_Name}")
print(f"     Design Flow Rate: {ghe.Design_Flow_Rate} m³/s")
print(f"     Ground: k={ghe.Ground_Thermal_Conductivity} W/(m·K), ρCp={ghe.Ground_Thermal_Heat_Capacity:.1e} J/(m³·K)")
print(f"     Ground Temperature Model: {ghe.Undisturbed_Ground_Temperature_Model_Name}")
print(f"     Response Factors: {ghe.GHEVerticalResponseFactors_Object_Name}")


  ✅ Vertical GHE System
     Nodes: GHE HX Inlet Node → GHE HX Outlet Node
     Design Flow Rate: 0.01344 m³/s
     Ground: k=1.8 W/(m·K), ρCp=2.2e+06 J/(m³·K)
     Ground Temperature Model: NYC Undisturbed Ground Temp
     Response Factors: GHE Response Factors


## Create GroundHeatExchanger:ResponseFactors (with g-function data)


In [55]:
rb_over_H = (0.1143 / 2) / 76.2

GFUNC_PAIRS = [
    (-15.2996,  1.0000), (-14.2010,  1.0220), (-13.2154,  1.0637),
    (-12.2310,  1.1518), (-11.2447,  1.3351), (-10.2599,  1.6762),
    ( -9.2740,  2.2237), ( -8.2864,  3.0029), ( -7.8000,  3.3956),
    ( -7.2000,  3.9044), ( -6.5000,  4.5555), ( -5.9000,  5.1530),
    ( -5.2000,  5.9527), ( -4.5000,  6.8860), ( -3.9630,  7.6857),
    ( -3.2700,  8.9150), ( -2.8640,  9.7095), ( -2.5770, 10.2610),
    ( -2.1720, 11.1280), ( -1.8840, 11.7440), ( -1.1910, 13.3190),
    ( -0.4970, 14.8670), (  0.1960, 16.3650), (  0.8900, 17.7870),
    (  1.5840, 19.0900), (  2.2780, 20.2200), (  2.9720, 21.1310),
    (  3.6650, 21.8030),
]

GHE_RF_IDF_TEXT = f"""GroundHeatExchanger:ResponseFactors,
    GHE Response Factors,            !- Name
    GHE Borehole Properties,         !- GHE:Vertical:Properties Object Name
    36,                              !- Number of Boreholes
    {rb_over_H:.6f},                     !- g-Function Reference Ratio (rb/H)
"""

for i, (lnts, g) in enumerate(GFUNC_PAIRS):
    ending = ";" if i == len(GFUNC_PAIRS) - 1 else ","
    GHE_RF_IDF_TEXT += f"    {lnts:10.4f},                        !- g-Function Ln(T/Ts) Value {i+1}\n"
    GHE_RF_IDF_TEXT += f"    {g:10.4f}{ending}                       !- g-Function g Value {i+1}\n"

print(f"✅ ResponseFactors IDF text ready ({len(GFUNC_PAIRS)} pairs of g-function)")
print(f"   rb/H = {rb_over_H:.6f}")
print(f"   Will be injected into the final IDF file")


✅ ResponseFactors IDF text ready (28 pairs of g-function)
   rb/H = 0.000750
   Will be injected into the final IDF file


## Create GHE Supply Side — Pump + Pipes

In [56]:
pump = idf.newidfobject("PUMP:VARIABLESPEED")
pump.Name                              = GHE["pump_name"]
pump.Inlet_Node_Name                   = GHE["supply_inlet"]
pump.Outlet_Node_Name                  = GHE["pump_outlet"]
pump.Design_Maximum_Flow_Rate          = "autosize"
pump.Design_Pump_Head                  = 179352
pump.Design_Power_Consumption          = "autosize"
pump.Motor_Efficiency                  = 0.9
pump.Fraction_of_Motor_Inefficiencies_to_Fluid_Stream = 0.0
pump.Coefficient_1_of_the_Part_Load_Performance_Curve = 0
pump.Coefficient_2_of_the_Part_Load_Performance_Curve = 1
pump.Coefficient_3_of_the_Part_Load_Performance_Curve = 0
pump.Coefficient_4_of_the_Part_Load_Performance_Curve = 0
pump.Design_Minimum_Flow_Rate         = 0.0
pump.Pump_Control_Type                = "Intermittent"
print(f"  ✅ {pump.Name}")

# ── Supply Pipe ──
pipe = idf.newidfobject("PIPE:ADIABATIC")
pipe.Name             = "GHE Supply Bypass Pipe"
pipe.Inlet_Node_Name  = GHE["supply_bypass_inlet"]
pipe.Outlet_Node_Name = GHE["supply_bypass_outlet"]
print(f"  ✅ {pipe.Name}")

# ── Supply Outlet Pipe ──
pipe = idf.newidfobject("PIPE:ADIABATIC")
pipe.Name             = "GHE Supply Outlet Pipe"
pipe.Inlet_Node_Name  = GHE["supply_mixer_outlet"]
pipe.Outlet_Node_Name = GHE["supply_outlet"]
print(f"  ✅ {pipe.Name}")

  ✅ GHE Pump
  ✅ GHE Supply Bypass Pipe
  ✅ GHE Supply Outlet Pipe


## Create GHE Supply Branches

In [57]:
supply_branches = [
    ("GHE Supply Inlet Branch",     "Pump:VariableSpeed",
     GHE["pump_name"],              GHE["supply_inlet"],         GHE["pump_outlet"]),

    ("GHE Supply Equipment Branch", "GroundHeatExchanger:System",
     GHE["ghe_name"],               GHE["ghe_inlet"],            GHE["ghe_outlet"]),

    ("GHE Supply Bypass Branch",    "Pipe:Adiabatic",
     "GHE Supply Bypass Pipe",      GHE["supply_bypass_inlet"],  GHE["supply_bypass_outlet"]),

    ("GHE Supply Outlet Branch",    "Pipe:Adiabatic",
     "GHE Supply Outlet Pipe",      GHE["supply_mixer_outlet"],  GHE["supply_outlet"]),
]

for br_name, comp_type, comp_name, inlet, outlet in supply_branches:
    br = idf.newidfobject("BRANCH")
    br.Name = br_name
    br.Component_1_Object_Type      = comp_type
    br.Component_1_Name             = comp_name
    br.Component_1_Inlet_Node_Name  = inlet
    br.Component_1_Outlet_Node_Name = outlet
    print(f"  ✅ {br.Name}")
    print(f"     [{comp_type}]: {inlet} → {outlet}")


  ✅ GHE Supply Inlet Branch
     [Pump:VariableSpeed]: GHE Supply Inlet Node → GHE Pump Outlet Node
  ✅ GHE Supply Equipment Branch
     [GroundHeatExchanger:System]: GHE HX Inlet Node → GHE HX Outlet Node
  ✅ GHE Supply Bypass Branch
     [Pipe:Adiabatic]: GHE Supply Bypass Inlet Node → GHE Supply Bypass Outlet Node
  ✅ GHE Supply Outlet Branch
     [Pipe:Adiabatic]: GHE Supply Mixer Outlet Node → GHE Supply Outlet Node


## Create GHE Supply — BranchList / Splitter / Mixer / ConnectorList

In [58]:
bl = idf.newidfobject("BRANCHLIST")
bl.Name          = "GHE Supply Branches"
bl.Branch_1_Name = "GHE Supply Inlet Branch"
bl.Branch_2_Name = "GHE Supply Equipment Branch"
bl.Branch_3_Name = "GHE Supply Bypass Branch"
bl.Branch_4_Name = "GHE Supply Outlet Branch"
print(f"  ✅ BranchList: {bl.Name}")

sp = idf.newidfobject("CONNECTOR:SPLITTER")
sp.Name                   = "GHE Supply Splitter"
sp.Inlet_Branch_Name      = "GHE Supply Inlet Branch"
sp.Outlet_Branch_1_Name   = "GHE Supply Equipment Branch"
sp.Outlet_Branch_2_Name   = "GHE Supply Bypass Branch"
print(f"  ✅ Splitter: {sp.Name}")

mx = idf.newidfobject("CONNECTOR:MIXER")
mx.Name                  = "GHE Supply Mixer"
mx.Outlet_Branch_Name    = "GHE Supply Outlet Branch"
mx.Inlet_Branch_1_Name   = "GHE Supply Equipment Branch"
mx.Inlet_Branch_2_Name   = "GHE Supply Bypass Branch"
print(f"  ✅ Mixer: {mx.Name}")

cl = idf.newidfobject("CONNECTORLIST")
cl.Name                    = "GHE Supply Connectors"
cl.Connector_1_Object_Type = "Connector:Splitter"
cl.Connector_1_Name        = "GHE Supply Splitter"
cl.Connector_2_Object_Type = "Connector:Mixer"
cl.Connector_2_Name        = "GHE Supply Mixer"
print(f"  ✅ ConnectorList: {cl.Name}")


  ✅ BranchList: GHE Supply Branches
  ✅ Splitter: GHE Supply Splitter
  ✅ Mixer: GHE Supply Mixer
  ✅ ConnectorList: GHE Supply Connectors


## Create GHE Demand Side — Pipes

In [59]:
pipe = idf.newidfobject("PIPE:ADIABATIC")
pipe.Name             = "GHE Demand Inlet Pipe"
pipe.Inlet_Node_Name  = GHE["demand_inlet"]
pipe.Outlet_Node_Name = GHE["demand_pipe_inlet_out"]
print(f"  ✅ {pipe.Name}")

pipe = idf.newidfobject("PIPE:ADIABATIC")
pipe.Name             = "GHE Demand Bypass Pipe"
pipe.Inlet_Node_Name  = GHE["demand_bypass_inlet"]
pipe.Outlet_Node_Name = GHE["demand_bypass_outlet"]
print(f"  ✅ {pipe.Name}")

pipe = idf.newidfobject("PIPE:ADIABATIC")
pipe.Name             = "GHE Demand Outlet Pipe"
pipe.Inlet_Node_Name  = GHE["demand_mixer_outlet"]
pipe.Outlet_Node_Name = GHE["demand_outlet"]
print(f"  ✅ {pipe.Name}")

  ✅ GHE Demand Inlet Pipe
  ✅ GHE Demand Bypass Pipe
  ✅ GHE Demand Outlet Pipe


## Create GHE Demand Branches: Connect WWHP Source Side

In [60]:
# ── Demand Inlet Branch ──
br = idf.newidfobject("BRANCH")
br.Name = "GHE Demand Inlet Branch"
br.Component_1_Object_Type      = "Pipe:Adiabatic"
br.Component_1_Name             = "GHE Demand Inlet Pipe"
br.Component_1_Inlet_Node_Name  = GHE["demand_inlet"]
br.Component_1_Outlet_Node_Name = GHE["demand_pipe_inlet_out"]
print(f"  ✅ {br.Name}")

# ──  WWHP Cooling HP — Source Side Branch  ──
br = idf.newidfobject("BRANCH")
br.Name = "GHE Demand CoolHP Source Branch"
br.Component_1_Object_Type      = "HeatPump:WaterToWater:EquationFit:Cooling"
br.Component_1_Name             = "GSHP Cooling HP"
br.Component_1_Inlet_Node_Name  = GHE["cool_hp_source_inlet"]
br.Component_1_Outlet_Node_Name = GHE["cool_hp_source_outlet"]
print(f"  ✅ {br.Name}  ★ WWHP Cooling Source Side")
print(f"     {GHE['cool_hp_source_inlet']} → {GHE['cool_hp_source_outlet']}")

# ──  WWHP Heating HP — Source Side Branch  ──
br = idf.newidfobject("BRANCH")
br.Name = "GHE Demand HeatHP Source Branch"
br.Component_1_Object_Type      = "HeatPump:WaterToWater:EquationFit:Heating"
br.Component_1_Name             = "GSHP Heating HP"
br.Component_1_Inlet_Node_Name  = GHE["heat_hp_source_inlet"]
br.Component_1_Outlet_Node_Name = GHE["heat_hp_source_outlet"]
print(f"  ✅ {br.Name}  ★ WWHP Heating Source Side")
print(f"     {GHE['heat_hp_source_inlet']} → {GHE['heat_hp_source_outlet']}")

# ── Demand Bypass Branch ──
br = idf.newidfobject("BRANCH")
br.Name = "GHE Demand Bypass Branch"
br.Component_1_Object_Type      = "Pipe:Adiabatic"
br.Component_1_Name             = "GHE Demand Bypass Pipe"
br.Component_1_Inlet_Node_Name  = GHE["demand_bypass_inlet"]
br.Component_1_Outlet_Node_Name = GHE["demand_bypass_outlet"]
print(f"  ✅ {br.Name}")

# ── Demand Outlet Branch ──
br = idf.newidfobject("BRANCH")
br.Name = "GHE Demand Outlet Branch"
br.Component_1_Object_Type      = "Pipe:Adiabatic"
br.Component_1_Name             = "GHE Demand Outlet Pipe"
br.Component_1_Inlet_Node_Name  = GHE["demand_mixer_outlet"]
br.Component_1_Outlet_Node_Name = GHE["demand_outlet"]
print(f"  ✅ {br.Name}")

  ✅ GHE Demand Inlet Branch
  ✅ GHE Demand CoolHP Source Branch  ★ WWHP Cooling Source Side
     GSHP CoolHP Source Inlet Node → GSHP CoolHP Source Outlet Node
  ✅ GHE Demand HeatHP Source Branch  ★ WWHP Heating Source Side
     GSHP HeatHP Source Inlet Node → GSHP HeatHP Source Outlet Node
  ✅ GHE Demand Bypass Branch
  ✅ GHE Demand Outlet Branch


## Create GHE Demand — BranchList / Splitter / Mixer / ConnectorList

In [61]:
bl = idf.newidfobject("BRANCHLIST")
bl.Name          = "GHE Demand Branches"
bl.Branch_1_Name = "GHE Demand Inlet Branch"
bl.Branch_2_Name = "GHE Demand CoolHP Source Branch"
bl.Branch_3_Name = "GHE Demand HeatHP Source Branch"
bl.Branch_4_Name = "GHE Demand Bypass Branch"
bl.Branch_5_Name = "GHE Demand Outlet Branch"
print(f"  ✅ BranchList: {bl.Name}")

sp = idf.newidfobject("CONNECTOR:SPLITTER")
sp.Name                   = "GHE Demand Splitter"
sp.Inlet_Branch_Name      = "GHE Demand Inlet Branch"
sp.Outlet_Branch_1_Name   = "GHE Demand CoolHP Source Branch"
sp.Outlet_Branch_2_Name   = "GHE Demand HeatHP Source Branch"
sp.Outlet_Branch_3_Name   = "GHE Demand Bypass Branch"
print(f"  ✅ Splitter: {sp.Name}")

mx = idf.newidfobject("CONNECTOR:MIXER")
mx.Name                  = "GHE Demand Mixer"
mx.Outlet_Branch_Name    = "GHE Demand Outlet Branch"
mx.Inlet_Branch_1_Name   = "GHE Demand CoolHP Source Branch"
mx.Inlet_Branch_2_Name   = "GHE Demand HeatHP Source Branch"
mx.Inlet_Branch_3_Name   = "GHE Demand Bypass Branch"
print(f"  ✅ Mixer: {mx.Name}")

cl = idf.newidfobject("CONNECTORLIST")
cl.Name                    = "GHE Demand Connectors"
cl.Connector_1_Object_Type = "Connector:Splitter"
cl.Connector_1_Name        = "GHE Demand Splitter"
cl.Connector_2_Object_Type = "Connector:Mixer"
cl.Connector_2_Name        = "GHE Demand Mixer"
print(f"  ✅ ConnectorList: {cl.Name}")


  ✅ BranchList: GHE Demand Branches
  ✅ Splitter: GHE Demand Splitter
  ✅ Mixer: GHE Demand Mixer
  ✅ ConnectorList: GHE Demand Connectors


## Create PlantLoop (Condenser)

In [62]:
pl = idf.newidfobject("PLANTLOOP")
pl.Name                                = GHE["loop_name"]
pl.Fluid_Type                          = "Water"
pl.Plant_Equipment_Operation_Scheme_Name = GHE["ops_scheme_name"]
pl.Loop_Temperature_Setpoint_Node_Name = GHE["supply_outlet"]
pl.Maximum_Loop_Temperature            = 40.0
pl.Minimum_Loop_Temperature            = -5.0
pl.Maximum_Loop_Flow_Rate              = "autosize"
pl.Minimum_Loop_Flow_Rate              = 0.0
pl.Plant_Loop_Volume                   = "autosize"
pl.Plant_Side_Inlet_Node_Name          = GHE["supply_inlet"]
pl.Plant_Side_Outlet_Node_Name         = GHE["supply_outlet"]
pl.Plant_Side_Branch_List_Name         = "GHE Supply Branches"
pl.Plant_Side_Connector_List_Name      = "GHE Supply Connectors"
pl.Demand_Side_Inlet_Node_Name         = GHE["demand_inlet"]
pl.Demand_Side_Outlet_Node_Name        = GHE["demand_outlet"]
pl.Demand_Side_Branch_List_Name        = "GHE Demand Branches"
pl.Demand_Side_Connector_List_Name     = "GHE Demand Connectors"
pl.Load_Distribution_Scheme            = "Optimal"

print(f"  ✅ {pl.Name}")
print(f"     Supply: {pl.Plant_Side_Inlet_Node_Name} → {pl.Plant_Side_Outlet_Node_Name}")
print(f"     Demand: {pl.Demand_Side_Inlet_Node_Name} → {pl.Demand_Side_Outlet_Node_Name}")
print(f"     Temp Range: {pl.Minimum_Loop_Temperature}°C ~ {pl.Maximum_Loop_Temperature}°C")


  ✅ GHE Loop
     Supply: GHE Supply Inlet Node → GHE Supply Outlet Node
     Demand: GHE Demand Inlet Node → GHE Demand Outlet Node
     Temp Range: -5.0°C ~ 40.0°C


## Sizing:Plant (Condenser)

In [63]:
sz = idf.newidfobject("SIZING:PLANT")
sz.Plant_or_Condenser_Loop_Name = GHE["loop_name"]
sz.Loop_Type                    = "Condenser"
sz.Design_Loop_Exit_Temperature = 30.0
sz.Loop_Design_Temperature_Difference = 5.5

print(f"  ✅ Sizing:Plant: {GHE['loop_name']}")
print(f"     Loop Type:    Condenser")
print(f"     Design Exit Temperature:  30.0°C")
print(f"     Design Temperature Difference:      5.5°C")

  ✅ Sizing:Plant: GHE Loop
     Loop Type:    Condenser
     Design Exit Temperature:  30.0°C
     Design Temperature Difference:      5.5°C


## PlantEquipmentOperationSchemes (Uncontrolled)

In [64]:
ops = idf.newidfobject("PLANTEQUIPMENTOPERATIONSCHEMES")
ops.Name = GHE["ops_scheme_name"]
ops.Control_Scheme_1_Object_Type   = "PlantEquipmentOperation:Uncontrolled"
ops.Control_Scheme_1_Name          = GHE["ops_uncontrolled_name"]
ops.Control_Scheme_1_Schedule_Name = "ALWAYS_ON"
print(f"  ✅ PlantEquipmentOperationSchemes: {ops.Name}")

op = idf.newidfobject("PLANTEQUIPMENTOPERATION:UNCONTROLLED")
op.Name = GHE["ops_uncontrolled_name"]
op.Equipment_List_Name = GHE["equip_list_name"]
print(f"  ✅ Uncontrolled Operation: {op.Name}")

el = idf.newidfobject("PLANTEQUIPMENTLIST")
el.Name = GHE["equip_list_name"]
el.Equipment_1_Object_Type = "GroundHeatExchanger:System"
el.Equipment_1_Name        = GHE["ghe_name"]
print(f"  ✅ PlantEquipmentList: {el.Name}")
print(f"     Equipment: {GHE['ghe_name']}")

  ✅ PlantEquipmentOperationSchemes: GHE Loop Operation Schemes
  ✅ Uncontrolled Operation: GHE Uncontrolled Operation
  ✅ PlantEquipmentList: GHE Equipment List
     Equipment: Vertical GHE System


## SetpointManager + Temperature Schedule

In [65]:
sch = idf.newidfobject("SCHEDULE:COMPACT")
sch.Name             = GHE["temp_sch_name"]
sch.Schedule_Type_Limits_Name = "Temperature"
sch.Field_1          = "Through: 12/31"
sch.Field_2          = "For: AllDays"
sch.Field_3          = "Until: 24:00,22.0"
print(f"  ✅ Schedule: {sch.Name} (22°C)")

spm = idf.newidfobject("SETPOINTMANAGER:SCHEDULED")
spm.Name                        = GHE["spm_name"]
spm.Control_Variable            = "Temperature"
spm.Schedule_Name               = GHE["temp_sch_name"]
spm.Setpoint_Node_or_NodeList_Name = GHE["supply_outlet"]
print(f"  ✅ SetpointManager: {spm.Name}")
print(f"     Node: {GHE['supply_outlet']}")

  ✅ Schedule: GHE Loop Temp Schedule (22°C)
  ✅ SetpointManager: GHE Loop Setpoint Manager
     Node: GHE Supply Outlet Node


# Save Final IDF


In [ ]:
FINAL_OUTPUT = r"C:\Users\yhw15\Documents\1947-RP\STD2022_NewYork_GSHP_v2\ASHRAE901_OfficeMedium_STD2022_NewYork_GSHP_FINAL.idf"

# Step 1: eppy saveas       
idf.saveas(FINAL_OUTPUT)
print(f"✅ Step 1: eppy saveas → {FINAL_OUTPUT}")

# Step 2: Text Write ResponseFactors
import re
with open(FINAL_OUTPUT, "r") as f:
    content = f.read()

pattern = r"GroundHeatExchanger:ResponseFactors,.*?;"
match = re.search(pattern, content, re.DOTALL | re.IGNORECASE)

if match:
    content = content[:match.start()] + GHE_RF_IDF_TEXT + content[match.end():]
    print("✅ Step 2: Replace ResponseFactors")
else:
    content = content.rstrip() + "\n\n" + GHE_RF_IDF_TEXT + "\n"
    print("✅ Step 2: Append ResponseFactors")

with open(FINAL_OUTPUT, "w") as f:
    f.write(content)

print(f"\n✅ Final IDF Saved: {FINAL_OUTPUT}")
print(f"   g-function: {len(GFUNC_PAIRS)}(6x6 Array, B/H=0.079, rb/H={rb_over_H:.6f})")
